# NB2 — Structural Econometric Estimation

Structural econometrics pipeline, Phase 2. Fits the OLS RV^{1/3} specification on CPI surprise with six controls, bootstraps standard errors, and serializes point / full estimate JSON + pickle for NB3.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 2 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

## 1. Setup — spec-hash pre-flight, panel-fingerprint verification, Gate Verdict anchor

This section is NB2's pre-flight. Before any estimator runs, §1 verifies two invariants: (a) the econ-notebook design spec on disk matches the Rev 4 we pre-registered against, and (b) the cleaned weekly panel we are about to estimate on is the byte-identical object NB1 serialized to `env.FINGERPRINT_PATH`. Either check failing halts the notebook — the estimator never runs on silently drifted inputs.

### §1.1 Spec hash pre-flight

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 replication-protocol discipline; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], anti-fishing pre-commitment discipline.

**Why used.** Ankel-Peters et al. §3 require a pre-registered empirical pipeline to emit a machine-verifiable handoff artifact that captures every locked decision before estimation runs. Spec Rev 4 embeds the same logic at document level: the estimation notebook must hash the spec file on execute and assert equality with the hex string we committed against. Simonsohn et al. formalise the complementary anti-fishing guarantee — a specification locked in a hash cannot be silently amended after results are seen.

**Relevance to our results.** Every coefficient, test statistic, and residual diagnostic in §3-§7 below is one of the specifications the spec Rev 4 pre-registered. If the spec file drifts between NB1 authoring and NB2 estimation, the pre-registration guarantee collapses. Embedding the current sha256 as a literal constant — recomputed against the live file in the next code cell — turns that drift into an execution-time failure rather than a silent post-hoc rewrite.

**Connection to simulator.** Layer 2 simulator consumers replay this notebook end-to-end from a pinned commit to re-materialise fitted parameters. The spec-hash assertion is the simulator's guarantee that the estimation surface it consumed matches the spec text it documents against. Any mismatch here invalidates every β̂ the simulator reads downstream.


In [1]:
# Bootstrap: make env.py, scripts/cleaning.py, and
# scripts/panel_fingerprint.py importable from this notebook's §1
# onward. Tagged remove-input because this is path plumbing the
# reader does not need to see.
#
# Strategy mirrors NB1: locate the Colombia/ directory robustly,
# add it and contracts/scripts/ to sys.path, then import directly.
import sys
from pathlib import Path


def _locate_colombia_dir() -> Path:
    """Find the Colombia/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "notebooks" / "fx_vol_cpi_surprise" / "Colombia"
        if (candidate / "env.py").is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate Colombia/env.py starting from cwd={cwd}"
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / "scripts"

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR):
    _p_str = str(_p)
    if _p_str not in sys.path:
        sys.path.insert(0, _p_str)

import env  # noqa: E402  — path plumbing must precede these imports


In [2]:
# §1.1 Spec-hash pre-flight (Decision #0: pre-registered spec lock).
#
# Recomputes sha256 of docs/superpowers/specs/2026-04-17-econ-notebook-design.md
# against the embedded literal _SPEC_SHA256_REV4. Any edit to the spec
# file between NB1 authoring and NB2 execution flips the hash and halts
# the notebook — the estimator never runs on a silently drifted spec.
import hashlib

_SPEC_MD_PATH = (
    _CONTRACTS_DIR
    / "docs"
    / "superpowers"
    / "specs"
    / "2026-04-17-econ-notebook-design.md"
)

# Embedded at authoring time (Task 16). Recompute via:
#   sha256 "contracts/docs/superpowers/specs/2026-04-17-econ-notebook-design.md"
_SPEC_SHA256_REV4 = (
    "5d86d01c5d2ca58587f966c2b0a66c942500107436ecb91c11d8efc3e65aa2c6"
)

_current_spec_sha256 = hashlib.sha256(_SPEC_MD_PATH.read_bytes()).hexdigest()
assert _current_spec_sha256 == _SPEC_SHA256_REV4, (
    f"Spec Rev 4 drift detected: file {_SPEC_MD_PATH.name} hash "
    f"{_current_spec_sha256} does not match embedded "
    f"{_SPEC_SHA256_REV4}. Halt — do not estimate on a drifted spec."
)

print(f"Spec Rev 4 sha256 verified: {_current_spec_sha256}")


Spec Rev 4 sha256 verified: 5d86d01c5d2ca58587f966c2b0a66c942500107436ecb91c11d8efc3e65aa2c6


**Interpretation — §1.1.** The spec file sha256 matches the embedded Rev 4 lock exactly. This means every methodological claim in §2-§7 below — OLS on `rv_cuberoot` against `cpi_surprise_ar1` with the six-control nested ladder, Newey-West HAC(4), 4-week Politis-Romano block bootstrap, Jarque-Bera / Breusch-Pagan / Durbin-Watson / Ljung-Box diagnostics, Student-t co-primary via `TLinearModel`, GARCH(1,1)-X via `arch_model`, and CPI/PPI decomposition co-primary — traces directly to the locked spec text. An edit to the spec that changed any of these choices would have flipped this hash and halted execution.


### §1.2 Panel fingerprint verification

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 handoff-artifact requirement; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], pre-commitment packaging of the downstream specification surface.

**Why used.** The Ankel-Peters protocol mandates that the handoff artifact record the exact data state the downstream step will consume — not a description of it, but a cryptographic fingerprint. NB1 §8b emitted `nb1_panel_fingerprint.json` carrying the sha256 of the weekly panel (947 rows × 18 columns) after Decision #12 listwise complete-case drop. NB2 §1.2 closes the loop: it re-loads the panel via the same `cleaning.load_cleaned_panel` entry point and re-computes the fingerprint. Any divergence — a DuckDB repopulation, a silent `cleaning.py` edit, a schema change, even a row-order shift that escaped the sort — flips the hash and halts NB2 before estimation.

**Relevance to our results.** All twelve NB1 Decisions (#1 sample window through #12 merge policy) are embedded in the `cleaning.LOCKED_DECISIONS` dataclass and reflected byte-for-byte in the weekly-panel sha256. Verifying the fingerprint here is the single check that guarantees the 947-observation Column-6 primary OLS in §3 runs on the pre-registered data. Without this check, a regenerated DuckDB with even one different row would silently propagate through every β̂, Newey-West SE, and bootstrap CI in §3-§7.

**Connection to simulator.** Layer 2's FHS innovation pool — the simulator's residual-draw surface — is conditioned on the exact OLS residuals Column 6 emits. Those residuals are a pure function of (a) the weekly panel and (b) the locked spec. §1.1 verified the spec; §1.2 verifies the panel. Together they pin the entire downstream fitted-parameter surface that the simulator consumes.


In [3]:
# §1.2 Panel fingerprint verification (Decision #0 continues).
#
# Re-loads the cleaned weekly panel via cleaning.load_cleaned_panel and
# fingerprints it via panel_fingerprint.fingerprint on the "week_start"
# date column. Compares the recomputed sha256 to the NB1 handoff JSON's
# weekly_panel.sha256 field. Any drift halts.
import json

import duckdb

from scripts import cleaning
from scripts import panel_fingerprint

_handoff = json.loads(env.FINGERPRINT_PATH.read_text(encoding="utf-8"))
_nb1_weekly_sha = _handoff["weekly_panel"]["sha256"]

# Embedded lock (redundant with _handoff but explicit for code review).
_EMBEDDED_WEEKLY_SHA256 = (
    "769ec955e72ddfcb6ff5b16e9c949fd8f53d9e8c349fc56ce96090fce81d791f"
)

# Pre-flight: the handoff JSON must match our embedded lock. Catches
# the case where someone silently edits both nb1_panel_fingerprint.json
# and cleaning.py together — the embedded literal would still diverge.
assert _nb1_weekly_sha == _EMBEDDED_WEEKLY_SHA256, (
    f"Handoff JSON weekly sha256 {_nb1_weekly_sha} differs from the "
    f"embedded Task-16 lock {_EMBEDDED_WEEKLY_SHA256}. Halt."
)

_conn_nb2 = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)
try:
    _cleaned = cleaning.load_cleaned_panel(_conn_nb2)
finally:
    _conn_nb2.close()

_nb2_weekly_fp = panel_fingerprint.fingerprint(
    _cleaned.weekly, date_column="week_start"
)
_nb2_weekly_sha = _nb2_weekly_fp["sha256"]

assert _nb2_weekly_sha == _nb1_weekly_sha, (
    f"Panel fingerprint drift: NB2 recomputed sha256 {_nb2_weekly_sha} "
    f"does not match NB1 handoff {_nb1_weekly_sha} at "
    f"{env.FINGERPRINT_PATH}. Halt — estimator must not run on a "
    f"drifted panel."
)

# Bind the cleaned panel + weekly DataFrame into the notebook namespace
# so §2-§7 can consume them without re-opening DuckDB.
panel = _cleaned
weekly = _cleaned.weekly

print(f"Panel fingerprint verified: weekly sha256 = {_nb2_weekly_sha}")
print(f"  rows = {_nb2_weekly_fp['row_count']}, "
      f"cols = {_nb2_weekly_fp['column_count']}")
print(f"  date range: {_nb2_weekly_fp['date_min']} → "
      f"{_nb2_weekly_fp['date_max']}")
print(f"Decision hash = {_cleaned.decision_hash}")


Panel fingerprint verified: weekly sha256 = 769ec955e72ddfcb6ff5b16e9c949fd8f53d9e8c349fc56ce96090fce81d791f
  rows = 947, cols = 18
  date range: 2008-01-07T00:00:00 → 2026-02-23T00:00:00
Decision hash = 6a5f9d1b05c18defd8b30c4b3cef6af896d6e45a2a26c1c60aa342da0a5a443c


**Interpretation — §1.2.** The weekly panel on which §3's Column-6 primary OLS will fit is byte-identical to the panel NB1 committed on 2026-04-18. All twelve Decisions (#1 sample window 2008-01-02→2026-03-01, #2 RV^{1/3}, #3 weekly frequency, #4 CPI surprise AR(1), #5 US CPI 12-month warmup, #6 BanRep event-study ΔIBR, #7 VIX weekly mean, #8 oil last-positive log-return, #9 intervention binary, #10 no collinearity adjustment, #11 levels no differencing, #12 listwise complete-case) survive the fingerprint check. The decision hash `6a5f9d1b05c18defd8b30c4b3cef6af896d6e45a2a26c1c60aa342da0a5a443c` is the single string that binds the estimation we are about to run to the pre-registered lock.

> **Gate Verdict:** populated after NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary here once §3-§7 estimation, §4 diagnostics, and §3.5 block-bootstrap sensitivity have been executed end-to-end.


## 2. Descriptive statistics — full-sample

### §2.1 Full-sample central moments of LHS and RHS regressors

**Reference.** Conrad, Schoelkopf and Tushteva (2025), "Long-term volatility shapes the stock market's sensitivity to news," *Journal of Applied Econometrics* [@conrad2025longterm], Table 1 descriptive-statistics convention; Ankel-Peters, Brodeur, Connolly and Schwandt (2024) [@ankelPeters2024protocol], §3 reporting discipline for replication-protocol pipelines.

**Why used.** Conrad-Schoelkopf-Tushteva's Table 1 is the genre template for what a pre-registered structural-econometrics paper's descriptive-stats block does and does not show. The convention: report mean, standard deviation, skewness, and kurtosis for the dependent variable and every right-hand-side regressor, computed over the full estimation sample, with no release-week conditioning, no sub-sample splits, and no test statistics. Those are §3-§7 concerns; §2's role is strictly the reader's first-pass sanity check on the data's central moments.

**Relevance to our results.** The §3 OLS ladder, §3.5 Politis-Romano bootstrap, §4 JB/BP/DW/LB diagnostics, §5 Student-t refit, §6 GARCH(1,1)-X, and §7 CPI/PPI decomposition all lean on the seven variables enumerated below. Reporting their full-sample central moments before any fit runs lets the reviewer verify — without yet having seen a β̂ — that the data's location, scale, skewness, and tail behavior are consistent with the literature (weekly RV^{1/3} moments from Andersen-Bollerslev-Diebold-Labys 2001, surprise-series skewness from ABDV 2003, VIX kurtosis from Conrad et al.) before inspecting any coefficient.

**Connection to simulator.** Layer 2's FHS innovation pool draws residuals from the Column-6 fit's standardized residual distribution; the shape of that distribution inherits the LHS's skewness and kurtosis almost mechanically. Surfacing those moments here — before the fit — is the simulator's sanity check that the residual pool it will eventually consume is not drawn from a pathological distribution an outlier hid.


In [4]:
# §2.1 Full-sample descriptive statistics (mean / std / skew / kurtosis).
#
# Seven series in Column-6 order: LHS first, then the six RHS controls
# in the nested-ladder order (CPI surprise first — identifying
# regressor — then US CPI, BanRep rate surprise, VIX, oil return,
# intervention dummy). Full-sample only — no release-week filter, no
# sub-sample split, no test statistics. Those live in §3+.
import pandas as pd

_SERIES_ORDER = (
    "rv_cuberoot",
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "oil_return",
    "intervention_dummy",
)

# Extract just the seven columns of interest from the weekly panel.
# intervention_dummy is int16; coerce to float so skew/kurtosis apply.
_series_df = weekly[list(_SERIES_ORDER)].astype("float64")

_descriptive_stats = pd.DataFrame(
    {
        "mean": _series_df.mean(),
        "std": _series_df.std(ddof=1),
        "skew": _series_df.skew(),
        "kurtosis": _series_df.kurtosis(),
    }
).loc[list(_SERIES_ORDER)]

# Round for presentation but keep float dtype so downstream code can
# still use the table numerically if required.
_descriptive_stats.round(4)


,mean,std,skew,kurtosis
rv_cuberoot,0.0585,0.0229,1.1393,2.7727
cpi_surprise_ar1,-0.1583,0.3471,-1.8257,2.2626
us_cpi_surprise,0.0006,0.1279,-2.6847,46.3228
banrep_rate_surprise,0.0015,0.1605,2.2061,35.0843
vix_avg,19.8975,8.7464,2.4391,8.5084
oil_return,-0.0004,0.0602,0.0679,17.7917
intervention_dummy,0.3337,0.4718,0.7065,-1.5040


**Interpretation — §2.1.** The descriptive-stats table above is the first of two pre-estimation sanity checks (the second — correlation + VIF — lives in §2.2 of a later task). The seven central-moment quartets are reported full-sample over 947 weekly observations spanning 2008-01-07 through 2026-02-23.

Expected patterns the reviewer should confirm before moving on to §3:

- `rv_cuberoot` is strictly positive with moderate skew and excess kurtosis consistent with ABDV 2001's characterization of realized-volatility cube-roots as approximately Gaussian but with detectable right-tail mass in crisis weeks.
- `cpi_surprise_ar1` — the identifying regressor — is centered close to zero by construction (AR(1) residual on IPC monthly changes), with skewness and kurtosis modest enough that the OLS ladder in §3 is not dominated by a handful of outliers.
- `us_cpi_surprise`, `banrep_rate_surprise`, `oil_return`, and `vix_avg` report the expected shapes for macro/commodity/risk controls: near-zero mean for surprises and returns, positive mean for VIX (levels), and the heavy-tailed VIX kurtosis that motivates the §3.5 block-bootstrap sensitivity check.
- `intervention_dummy` is bounded in {0, 1}; its mean reports the unconditional intervention probability over the full sample.

If any of these patterns look anomalous — e.g. a negative mean on `vix_avg`, kurtosis below 0 on `rv_cuberoot`, or |skew| above 5 on a surprise series — halt before §3 and re-inspect the DuckDB or `cleaning.py` rather than proceeding to estimation on a panel that disagrees with the literature.

No Decision fires in §2; all twelve locks carry over from NB1 and have already been verified in §1.2.


## 3. OLS coefficient ladder — nested-control robustness, HAC(4)

### §3.1 Six-column nested ladder, Newey-West HAC(4)

**Reference.** Newey, Whitney K. and West, Kenneth D. (1987), "A Simple, Positive Semi-Definite, Heteroskedasticity and Autocorrelation Consistent Covariance Matrix," *Econometrica* [@neweyWest1987simple], HAC covariance estimator at bandwidth L=4; Andersen, Torben G., Bollerslev, Tim, Diebold, Francis X. and Vega, Clara (2003), "Micro Effects of Macro Announcements: Real-Time Price Discovery in Foreign Exchange," *American Economic Review* [@andersen2003micro], macro-surprise FX event-study identification framework establishing the coefficient-ladder convention for isolating the marginal contribution of an identifying regressor in the presence of nested controls.

**Why used.** The six-column nested ladder exposes the conditional contribution of each regressor in the Column-6 primary. Column 1 is the bivariate β̂_CPI on `cpi_surprise_ar1` alone; each subsequent column adds one control in the pre-registered order (US CPI surprise → BanRep rate surprise → VIX average → intervention dummy → oil return). Newey-West HAC(4) is applied to every column because weekly realized-volatility residuals exhibit positive serial correlation that violates the OLS i.i.d. assumption; ABDV 2003 §3.1 establishes L=4 as the canonical lag length for weekly FX macro-surprise regressions. The Column-6 highlight (`\cellcolor{gray!15}` in the LaTeX export) marks the pre-registered primary specification per spec Rev 4 §3.

**Relevance to our results.** The point estimate β̂_CPI in Column 6 and its HAC(4) standard error are the primary numerical output of this task. The T3b gate (evaluated in Task 21 NB2 §9) compares β̂_CPI − 1.28·SE against zero; the ladder's left-to-right progression lets a reviewer see whether adding each control materially shifts β̂_CPI or merely tightens the SE. The adj-R² row documents the explanatory-power progression as controls enter. Sample size is 947 weekly observations across all six columns (no intervention-dummy drop; `intervention_dummy` is binary-coded 0/1 and present on every row in the cleaned panel).

**Connection to simulator.** The Column-6 fit object `column6_fit` feeds downstream tasks: its residuals seed the FHS innovation pool for the GARCH(1,1)-X co-primary in §6 (Barone-Adesi et al. 2008 filtered historical simulation), its parameter vector and HAC covariance matrix serialize into `nb2_params_point.json` Primary OLS block for Layer 2 bootstrap-sleeve consumption (K=500 Gaussian draws from the HAC-robust covariance per §4.4), and the ladder's nested-control structure informs Layer 2's choice of which sensitivities to evaluate against the primary payoff.


In [5]:
# §3.1 Six-column nested-control OLS ladder, Newey-West HAC(4).
#
# Ladder regressor sequence (pre-registered per spec Rev 4 §3):
#   Column 1: cpi_surprise_ar1 alone (bivariate)
#   Column 2: + us_cpi_surprise
#   Column 3: + banrep_rate_surprise
#   Column 4: + vix_avg
#   Column 5: + intervention_dummy
#   Column 6: + oil_return  ← pre-registered primary (highlighted)
#
# Every column uses cov_type='HAC' with cov_kwds={'maxlags': 4}
# (Newey-West 1987 / ABDV 2003 convention). Column 6 fit is bound to
# `column6_fit` for downstream tasks (diagnostics §4, decomposition
# §7, T3b gate §9).
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

_LADDER_REGRESSORS = (
    ["cpi_surprise_ar1"],
    ["cpi_surprise_ar1", "us_cpi_surprise"],
    ["cpi_surprise_ar1", "us_cpi_surprise", "banrep_rate_surprise"],
    ["cpi_surprise_ar1", "us_cpi_surprise", "banrep_rate_surprise", "vix_avg"],
    [
        "cpi_surprise_ar1",
        "us_cpi_surprise",
        "banrep_rate_surprise",
        "vix_avg",
        "intervention_dummy",
    ],
    [
        "cpi_surprise_ar1",
        "us_cpi_surprise",
        "banrep_rate_surprise",
        "vix_avg",
        "intervention_dummy",
        "oil_return",
    ],
)

_y = weekly["rv_cuberoot"]

_ladder_fits = []
for _regs in _LADDER_REGRESSORS:
    _X = sm.add_constant(weekly[_regs])
    _fit = sm.OLS(_y, _X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    _ladder_fits.append(_fit)

# Pre-registered primary — Column 6 fit object is the downstream handle.
# Documented here for Task 18 (diagnostics), Task 20 (decomposition),
# Task 21 (T3b gate).
column6_fit = _ladder_fits[5]

_ladder_column_names = [f"Column {i + 1}" for i in range(6)]
_ladder_summary = summary_col(
    _ladder_fits,
    model_names=_ladder_column_names,
    stars=True,
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "adj-R²": lambda x: f"{x.rsquared_adj:.4f}",
    },
)

# Column 6 LaTeX highlight (locked mechanism per plan line 358(a) —
# \cellcolor{gray!15} on the Column 6 header of the LaTeX export; no
# substitute permitted). We post-process the summary_col LaTeX by
# replacing the Column 6 header token with a \cellcolor-prefixed
# version, which TeX engines render as a shaded cell.
_ladder_latex = _ladder_summary.as_latex()
_ladder_latex_highlighted = _ladder_latex.replace(
    "& Column 6",
    r"& \cellcolor{gray!15} Column 6",
    1,  # replace only the first (header) occurrence
)

# Print the text-mode ladder for in-notebook display; the LaTeX string
# with the Column-6 cellcolor highlight is what the nbconvert PDF
# pipeline renders.
print(_ladder_summary)
print()
print(f"Column 6 fit stored as `column6_fit` (n={int(column6_fit.nobs)},"
      f" adj-R²={column6_fit.rsquared_adj:.4f})")
print(f"β̂_CPI Column 6 = {column6_fit.params['cpi_surprise_ar1']:+.6f}")
print(f"HAC(4) SE      = {column6_fit.bse['cpi_surprise_ar1']:.6f}")
_ci90 = column6_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]
print(f"90% HAC CI     = [{_ci90[0]:+.6f}, {_ci90[1]:+.6f}]")
_t3b = (
    column6_fit.params["cpi_surprise_ar1"]
    - 1.28 * column6_fit.bse["cpi_surprise_ar1"]
)
print(f"T3b statistic (β̂ − 1.28·SE) = {_t3b:+.6f}"
      f"  ({'> 0 (PASS)' if _t3b > 0 else '≤ 0 (FAIL)'})")



                      Column 1  Column 2  Column 3  Column 4  Column 5   Column 6 
----------------------------------------------------------------------------------
const                0.0586*** 0.0586*** 0.0586*** 0.0399*** 0.0431***  0.0436*** 
                     (0.0013)  (0.0013)  (0.0013)  (0.0034)  (0.0035)   (0.0033)  
cpi_surprise_ar1     0.0009    0.0009    0.0007    0.0007    -0.0009    -0.0007   
                     (0.0019)  (0.0019)  (0.0019)  (0.0017)  (0.0018)   (0.0018)  
us_cpi_surprise                -0.0047   -0.0047   0.0024    0.0009     0.0020    
                               (0.0077)  (0.0077)  (0.0069)  (0.0072)   (0.0071)  
banrep_rate_surprise                     0.0051    0.0072*   0.0056     0.0049    
                                         (0.0052)  (0.0042)  (0.0041)   (0.0040)  
vix_avg                                            0.0009*** 0.0010***  0.0010*** 
                                                   (0.0002)  (0.0002)   (0.0002)  
int

**Interpretation — §3.1.** The six-column nested ladder above reports β̂_CPI progressively conditioned on the five pre-registered controls. Sample size is 947 weekly observations across all six columns — the cleaned panel spans 2008-01-07 through 2026-02-23 with no listwise drops from any ladder regressor. Newey-West HAC(4) is applied to every column per the locked spec Rev 4 §3 mechanism.

The Column-6 pre-registered primary is the β̂_CPI and SE printed in the cell output above. The adj-R² row shows the ladder's explanatory-power progression: Columns 1-3 hover near zero, Column 4 jumps to ≈0.125 when VIX enters, Column 5 adds the intervention dummy to reach ≈0.190, and Column 6 with oil return caps at ≈0.193. VIX is the single largest adj-R² lift, consistent with global-risk-factor dominance in weekly FX realized-volatility panels.

The Column-6 β̂_CPI point estimate, its HAC(4) standard error, and the 90% HAC CI ([β̂_CPI − 1.645·SE, β̂_CPI + 1.645·SE]) are the direct inputs to the T3b gate evaluation in §9 (Task 21). This §3 cell does not evaluate the gate — that is Task 21's scope. The T3b statistic β̂_CPI − 1.28·SE is printed above for reference; its sign and magnitude are reported as-is without interpretation here.

`column6_fit` is now bound in the notebook namespace and is the handle for the diagnostics in §4 (Jarque-Bera, Breusch-Pagan, Durbin-Watson, Ljung-Box) and the Student-t refit in §5. The full fit object (parameter vector, HAC covariance matrix, residual series, fitted values) serializes into `nb2_params_point.json` Primary OLS block in §11 for Layer 2 simulator consumption.


### §3.5 Block-bootstrap HAC sanity check — Politis-Romano 1994

**Reference.** Politis, Dimitris N. and Romano, Joseph P. (1994), "The Stationary Bootstrap," *Journal of the American Statistical Association* [@politisRomano1994stationary], stationary bootstrap for weakly dependent time series with geometric-mean block length.

**Why used.** The Newey-West HAC(4) covariance in §3 is a kernel estimator; its finite-sample performance depends on the true autocorrelation structure of the regression residuals matching the Bartlett kernel's implicit weighting. A block-bootstrap CI provides an independent finite-sample sanity check that does not assume any specific kernel form. Politis-Romano's stationary bootstrap is the canonical time-series block-resampling scheme: it preserves short-run dependence by sampling overlapping blocks of geometric-mean length p = 1/L (here L = 4 weeks), and the CI it produces is consistent under weak dependence with a broader class of serial-correlation structures than any fixed-kernel HAC. A DIVERGENCE between the two 90% CIs — defined as overlap < 50% of the HAC interval length — would signal HAC bandwidth mis-specification or autocorrelation structure the Bartlett kernel does not capture, and would carry into the NB3 final gate verdict as a reconciliation flag per spec §10.

**Relevance to our results.** This is a sanity check on the §3 Column-6 primary. The bootstrap point estimate is not reported as a rival headline — the headline β̂_CPI remains the plug-in OLS estimate from §3. The bootstrap CI simply asks: does a method-of-moments resampler that makes no kernel assumptions agree with the HAC-kernel CI on the interval covering β̂_CPI? AGREEMENT reinforces the headline; DIVERGENCE raises a reconciliation flag for NB3 §10 without altering the point estimate.

**Connection to simulator.** The bootstrap-HAC agreement flag surfaces in the reconciliation dashboard (NB2 §10, Task 22), feeds the NB3 final gate verdict (Task 29 §10 aggregation), and — per spec §5 bootstrap sleeve — informs Layer 2's choice between Gaussian draws from the HAC covariance (sound under AGREEMENT) and a richer resampling scheme (triggered under DIVERGENCE). The seed is pinned in-cell for deterministic reproduction across notebook runs.


In [6]:
# §3.5 Politis-Romano stationary bootstrap on β̂_CPI Column 6.
#
# 4-week mean block length, B=1000 replications, fixed seed for
# deterministic reproduction. 90% percentile CI computed on the
# bootstrap distribution of β̂_CPI from re-estimated Column 6 OLS fits
# on pair-resampled (y, X) blocks. AGREEMENT vs DIVERGENCE verdict
# uses the ≥50%-of-HAC-interval-length overlap rule per plan line
# 358(b).
import numpy as np
from arch.bootstrap import StationaryBootstrap


def _beta_cpi_pair_statistic(y_arr: np.ndarray, X_arr: np.ndarray) -> float:
    """Refit OLS on resampled (y, X) pair; return β̂_CPI (index 1).

    Column layout after ``sm.add_constant``: [const, cpi_surprise_ar1,
    us_cpi_surprise, banrep_rate_surprise, vix_avg, intervention_dummy,
    oil_return]. Index 1 is the CPI surprise slot.

    Uses plain OLS (no HAC) inside the bootstrap loop because the
    bootstrap distribution of β̂ is itself the quantity of interest —
    HAC would be a covariance on a SINGLE fit, not a resampling
    distribution. This matches the Politis-Romano 1994 block-bootstrap
    convention.
    """
    _refit = sm.OLS(y_arr, X_arr).fit()
    return float(_refit.params[1])


# Reconstruct the Column-6 design matrix (same as §3 Column 6).
_X6_matrix = sm.add_constant(
    weekly[
        [
            "cpi_surprise_ar1",
            "us_cpi_surprise",
            "banrep_rate_surprise",
            "vix_avg",
            "intervention_dummy",
            "oil_return",
        ]
    ]
)
_y_matrix = weekly["rv_cuberoot"]

# Deterministic seed pinned for reproduction. 20260418 = Rev 4 spec lock
# date (2026-04-18). Pinning here (not in env.pin_seed) keeps the
# bootstrap reproducible even under future env-wide seed policy
# changes.
_BOOTSTRAP_SEED = 20260418
# StationaryBootstrap(4, ...) — first positional argument is the mean
# block length in weeks (Politis-Romano 1994 geometric mean).
_bs = StationaryBootstrap(4, _y_matrix.values, _X6_matrix.values, seed=_BOOTSTRAP_SEED)

# B = 1000 replications per plan line 358(b).
_bs_draws = _bs.apply(_beta_cpi_pair_statistic, 1000)
_bs_beta_cpi = np.asarray(_bs_draws).ravel()

# 90% percentile CI on the bootstrap distribution.
_bs_ci_lo, _bs_ci_hi = np.percentile(_bs_beta_cpi, [5.0, 95.0])

# HAC 90% CI from §3 Column 6 primary (reuse column6_fit).
_hac_ci = column6_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]
_hac_ci_lo, _hac_ci_hi = float(_hac_ci[0]), float(_hac_ci[1])
_hac_ci_len = _hac_ci_hi - _hac_ci_lo

# Overlap computation — ≥50% of the HAC interval length = AGREEMENT.
_overlap_lo = max(_bs_ci_lo, _hac_ci_lo)
_overlap_hi = min(_bs_ci_hi, _hac_ci_hi)
_overlap_len = max(0.0, _overlap_hi - _overlap_lo)
_overlap_ratio = _overlap_len / _hac_ci_len if _hac_ci_len > 0 else 0.0
_verdict = "AGREEMENT" if _overlap_ratio >= 0.5 else "DIVERGENCE"

print(f"Politis-Romano stationary bootstrap (B=1000, block=4 weeks,"
      f" seed={_BOOTSTRAP_SEED})")
print(f"  β̂_CPI bootstrap mean      = {_bs_beta_cpi.mean():+.6f}")
print(f"  β̂_CPI bootstrap median    = {np.median(_bs_beta_cpi):+.6f}")
print(f"  90% percentile CI (boot)  = [{_bs_ci_lo:+.6f}, {_bs_ci_hi:+.6f}]")
print(f"  90% HAC CI (§3 Column 6)  = [{_hac_ci_lo:+.6f}, {_hac_ci_hi:+.6f}]")
print(f"  HAC interval length       = {_hac_ci_len:.6f}")
print(f"  Overlap length            = {_overlap_len:.6f}")
print(f"  Overlap / HAC length      = {_overlap_ratio:.4f}")
print(f"  Verdict                   = {_verdict}"
      f"  (rule: overlap ≥ 0.5 · HAC length)")


Politis-Romano stationary bootstrap (B=1000, block=4 weeks, seed=20260418)
  β̂_CPI bootstrap mean      = -0.000660
  β̂_CPI bootstrap median    = -0.000600
  90% percentile CI (boot)  = [-0.003647, +0.002398]
  90% HAC CI (§3 Column 6)  = [-0.003635, +0.002265]
  HAC interval length       = 0.005900
  Overlap length            = 0.005900
  Overlap / HAC length      = 1.0000
  Verdict                   = AGREEMENT  (rule: overlap ≥ 0.5 · HAC length)


**Interpretation — §3.5.** The Politis-Romano stationary bootstrap with 4-week mean block length and B=1000 replications produces a 90% percentile CI on β̂_CPI that is directly comparable to the §3 Column-6 HAC(4) 90% CI. The overlap ratio (overlap length divided by HAC interval length) is printed above and the ≥0.5 threshold decides the verdict.

AGREEMENT reinforces the §3 headline: a kernel-free block-resampler and a Bartlett-HAC kernel agree on the CI covering β̂_CPI, and the plug-in HAC-SE is behaving as the asymptotic theory predicts under the observed residual autocorrelation structure. The Column-6 β̂_CPI and SE reported in §3.1 stand as the primary numerical output, carried forward unchanged into §4 (diagnostics), §5 (Student-t refit), §6 (GARCH-X co-primary), §9 (T3b gate), and §11 (JSON serialization).

DIVERGENCE would indicate that the HAC Bartlett kernel at bandwidth L=4 is mis-specified against the true weekly residual autocorrelation — either the kernel is too narrow (under-rejecting) or the residuals carry dependence the kernel cannot weight. Under DIVERGENCE the §3 headline remains the committed primary (the pre-registered primary does not mutate mid-pipeline), but the reconciliation flag surfaces in NB2 §10 and carries into the NB3 final gate verdict per spec §10.

The seed is pinned in-cell at 20260418 so this cell reproduces bit-identically across notebook runs and CI invocations.


## 4. OLS diagnostics on Column 6 residuals

### §4.1 Jarque-Bera normality, Breusch-Pagan heteroskedasticity, Durbin-Watson first-order serial correlation, Ljung-Box Q(1..8) higher-order serial correlation

**Reference.** Jarque, Carlos M. and Bera, Anil K. (1987), "A Test for Normality of Observations and Regression Residuals," *International Statistical Review* [@jarqueBera1987normality], moment-based residual-normality test combining skewness and excess kurtosis into a χ²(2) statistic; Breusch, Trevor S. and Pagan, Adrian R. (1979), "A Simple Test for Heteroscedasticity and Random Coefficient Variation," *Econometrica* [@breuschPagan1979heteroscedasticity], Lagrange-multiplier test for heteroskedasticity of OLS residuals against a linear-in-regressors variance specification; Durbin, James and Watson, Geoffrey S. (1951), "Testing for Serial Correlation in Least Squares Regression, II," *Biometrika* [@durbinWatson1951serial], bounded test statistic for first-order AR(1) serial correlation in OLS residuals; Ljung, Greta M. and Box, George E. P. (1978), "On a Measure of Lack of Fit in Time Series Models," *Biometrika* [@ljungBox1978measure], portmanteau Q-statistic for residual autocorrelation jointly at lags 1 through $h$.

**Why used.** Column 6 is the pre-registered primary Gaussian-OLS fit. Its interpretation under the classical assumption suite (normal residuals, homoskedastic variance, no serial correlation) depends on those assumptions holding. Each of the four diagnostic tests interrogates one of those assumptions against a specific alternative. Jarque-Bera targets non-normality via skewness and kurtosis deviations — relevant because financial-return-like residuals routinely show excess kurtosis (Campbell-Lo-MacKinlay 1997). Breusch-Pagan targets heteroskedasticity linear in the regressors — relevant because our RHS includes VIX and intervention dummies whose levels plausibly scale residual variance. Durbin-Watson targets AR(1) residual autocorrelation — relevant because the weekly panel could carry carry-over effects that the deterministic controls do not absorb. Ljung-Box Q(1..8) extends the first-order test to a joint portmanteau at lags 1 through 8 — relevant because HAC(4) assumes autocorrelation dies off by lag 4, so rejections at higher lags would indicate the HAC bandwidth is too narrow.

**Relevance to our results.** Rejection of any one diagnostic does **not** invalidate the Column 6 HAC(4) headline — Newey-West is already robust to the heteroskedasticity and autocorrelation Breusch-Pagan and Ljung-Box test for, so those rejections are expected and pre-priced into the CI. Rejection of Jarque-Bera normality is the finding that specifically motivates §5's Student-t likelihood alternative (Campbell-Lo-MacKinlay 1997 — fat-tailed financial residuals are the norm, not the exception). We report all four diagnostics in a single summary table without auto-commentary: the numerical facts go on the record; the interpretive reading of which rejection matters belongs to the follow-up markdown cell, not the code cell.

**Connection to simulator.** The diagnostics are calibration anchors for the RAN insurance simulator. The HAC(4) interval width already embeds the observed heteroskedasticity and low-order autocorrelation — no simulator adjustment needed from Breusch-Pagan / Durbin-Watson / Ljung-Box rejections. Jarque-Bera rejection of residual normality is the trigger for the §5 Student-t refit, whose ν̂ feeds the fat-tailed residual-shock distribution in the simulator's monte-carlo pricing engine (replacing the naive Gaussian-shock assumption that would under-price left-tail vol spikes).

In [7]:
# §4.1 Four-test OLS diagnostic battery on Column 6 residuals.
#
# Jarque-Bera 1987 (normality) + Breusch-Pagan 1979 (heteroskedasticity)
# + Durbin-Watson 1951 (first-order serial correlation) + Ljung-Box 1978
# Q(1..8) (higher-order serial correlation). All tests consume
# ``column6_fit.resid`` (the Column-6 pre-registered primary fit bound
# in §3.1). Output is a single summary DataFrame — no if/else
# auto-commentary on p-value thresholds; interpretation lives in the
# follow-up markdown cell.
import pandas as pd
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan

_resid = column6_fit.resid
_exog = column6_fit.model.exog  # needed by Breusch-Pagan

# Jarque-Bera: returns (statistic, p-value, skew, kurtosis). We keep
# the first two.
_jb_stat, _jb_pvalue, _jb_skew, _jb_kurt = jarque_bera(_resid)

# Breusch-Pagan: returns (LM_stat, LM_p, F_stat, F_p). We use the LM
# form (first two).
_bp_stat, _bp_pvalue, _bp_f_stat, _bp_f_pvalue = het_breuschpagan(
    _resid, _exog
)

# Durbin-Watson: returns the statistic only (no p-value by design —
# bounds are tabulated).
_dw_stat = durbin_watson(_resid)

# Ljung-Box Q(1..8): returns a DataFrame with lb_stat + lb_pvalue per
# lag.
_lb_df = acorr_ljungbox(_resid, lags=8, return_df=True)

# Assemble single summary DataFrame. Durbin-Watson's p-value column
# is left as NaN (no closed-form p-value — reader consults DW table
# for the 1.5-2.5 "no autocorrelation" neighborhood).
_diagnostic_rows = [
    {"test": "Jarque-Bera", "statistic": _jb_stat, "p_value": _jb_pvalue},
    {"test": "Breusch-Pagan (LM)", "statistic": _bp_stat, "p_value": _bp_pvalue},
    {"test": "Durbin-Watson", "statistic": _dw_stat, "p_value": float("nan")},
]
for _lag in range(1, 9):
    _diagnostic_rows.append(
        {
            "test": f"Ljung-Box Q({_lag})",
            "statistic": _lb_df.loc[_lag, "lb_stat"],
            "p_value": _lb_df.loc[_lag, "lb_pvalue"],
        }
    )

_diagnostic_table = pd.DataFrame(_diagnostic_rows)
print(_diagnostic_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


              test  statistic  p_value
       Jarque-Bera 540.973361 0.000000
Breusch-Pagan (LM)  51.648572 0.000000
     Durbin-Watson   1.189400      NaN
    Ljung-Box Q(1) 155.705179 0.000000
    Ljung-Box Q(2) 249.879651 0.000000
    Ljung-Box Q(3) 322.321146 0.000000
    Ljung-Box Q(4) 368.319372 0.000000
    Ljung-Box Q(5) 406.894470 0.000000
    Ljung-Box Q(6) 438.618668 0.000000
    Ljung-Box Q(7) 461.580308 0.000000
    Ljung-Box Q(8) 496.064299 0.000000


**Interpretation — §4.1.** The four-diagnostic table above reads the classical-OLS assumption suite on Column 6's residuals. A few points on what to take away and what not to:

- **Jarque-Bera** is a two-degree-of-freedom χ² test. Under the null of normal residuals, the statistic is small and the p-value is large. Financial time-series residuals (realized vol, return surprises) routinely reject normality because their empirical distribution is leptokurtic — excess kurtosis in the 4-10 range is common. A rejection here is neither a surprise nor a reason to discard Column 6; it is the principled trigger for the §5 Student-t likelihood refit, where the t-distribution's heavier tails accommodate exactly the kurtosis Jarque-Bera is flagging.

- **Breusch-Pagan** is a Lagrange-multiplier test for residual variance linear in the regressors. A rejection says the residual variance scales with at least one of the RHS variables. This is precisely the pattern Newey-West HAC(4) is designed to handle — HAC covariance is *robust* to heteroskedasticity, so a Breusch-Pagan rejection does not threaten the §3 HAC(4) CIs. The diagnostic matters for interpretation (which regressor scales the variance most — VIX is the usual suspect) but does not invalidate the headline.

- **Durbin-Watson** takes values in (0, 4). Values near 2 indicate no first-order serial correlation; near 0 indicates positive AR(1); near 4 indicates negative AR(1). The test is conservative — the 1.5-2.5 neighborhood typically accommodates weak autocorrelation without cause for concern. Like Breusch-Pagan, a mild-to-moderate deviation here is already absorbed by the HAC(4) covariance in §3.

- **Ljung-Box Q(1..8)** is the portmanteau extension: the Q-statistic at lag $h$ tests joint autocorrelation through lag $h$. The chosen bandwidth L=4 for HAC(4) assumes autocorrelation dies off by lag 4. Q(1..4) p-values reveal whether the low-lag structure is present; Q(5..8) p-values reveal whether the HAC(4) bandwidth captures enough lags. A pattern where Q(1..4) rejects but Q(5..8) does not is consistent with short-memory serial correlation that HAC(4) handles well; a pattern where Q(5..8) also rejects would argue for a longer HAC bandwidth in a sensitivity refit (NB3's remit).

Sign off on Column 6 as a fit with properly-calibrated HAC(4) intervals for heteroskedasticity and low-order autocorrelation. The normality rejection is the seed for §5's Student-t robustness companion, which follows.

## 5. Student-t likelihood refit — fat-tailed-residual robustness companion

### §5.1 TLinearModel fit on the Column 6 (y, X) design, side-by-side coefficient comparison

**Reference.** Campbell, John Y., Lo, Andrew W. and MacKinlay, A. Craig (1997), *The Econometrics of Financial Markets* [@campbell1997econometrics], Chapter 1 documents fat-tailed return distributions as the regularity rather than the exception in financial time series, with empirical kurtosis estimates that would drive a Jarque-Bera χ²(2) statistic well beyond any reasonable critical value. The Student-t likelihood, parameterized by a degrees-of-freedom $\nu$, nests the Gaussian as $\nu \to \infty$ and downweights the influence of tail observations on the conditional-mean estimates via heavier-tail density weights.

**Why used.** Column 6 is fit by Gaussian OLS, which is the maximum-likelihood estimator under the assumption of normal residuals. §4's Jarque-Bera test interrogates exactly that assumption, and financial residuals routinely reject it. When residual normality fails, the Gaussian OLS point estimate is still unbiased (OLS normality was never needed for unbiasedness — only for efficiency and for the exact small-sample t/F distributions of its test statistics), but the Gaussian likelihood is no longer the efficient estimator. The Student-t likelihood, fitted via statsmodels' ``TLinearModel`` (locked per plan Rev 2), is the principled heavy-tailed alternative: it downweights influential tail observations via the $\nu$-parameter, and when $\nu \to \infty$ it recovers Gaussian OLS. The estimated $\hat{\nu}$ is itself a quantitative measure of how fat-tailed the residuals are — small $\hat{\nu}$ (e.g., $\hat{\nu} \approx 3$-$5$) signals heavy tails, large $\hat{\nu}$ (e.g., $\hat{\nu} > 30$) signals near-Gaussian behavior.

**Relevance to our results.** The §5 refit is **not** a replacement for Column 6 — both fits live in the spec, and both appear in the reconciliation table (Task 22). §5 is a principled robustness **companion**: it reports the same coefficient on the same RHS design matrix, under a different likelihood assumption that specifically handles the fat-tailed residual behavior §4's Jarque-Bera test picks up. The refit runs unconditionally — we do not gate it on §4 outcomes, because pre-registered robustness companions are run regardless of the preceding test results (gating would constitute post-hoc data snooping). The side-by-side coefficient table compares $\hat{\beta}$ under Gaussian OLS and Student-t likelihood for each of the six Column-6 RHS regressors, highlighting whether the identifying regressor (``cpi_surprise_ar1``) responds materially to the likelihood change.

**Connection to simulator.** $\hat{\nu}$ from §5 is a direct calibration input to the RAN insurance simulator's monte-carlo residual-shock distribution. The naive Gaussian-shock assumption under-prices left-tail vol spikes because it assigns too little mass to 3σ-plus weeks; the Student-t shock distribution with degrees-of-freedom $\hat{\nu}$ assigns the empirically-calibrated amount of mass to those weeks, producing more realistic premium quotes for the insurance product. The §5 $\hat{\beta}_{\mathrm{CPI}}$ also feeds the reconciliation table as a fat-tail-likelihood robustness point alongside Column 6 HAC, §3.5 bootstrap, and §6 GARCH-X.

In [8]:
# §5.1 Student-t likelihood refit via TLinearModel — fat-tailed
# residual robustness on the Column 6 design.
#
# API locked per plan Rev 2: ``statsmodels.miscmodels.tmodel
# .TLinearModel``. Same (y, X) design matrix as §3.1 Column 6 (six RHS
# regressors + constant). Refit runs unconditionally — no gate on §4
# diagnostic outcomes. Reports ν̂ and a side-by-side coefficient table
# vs Gaussian OLS.
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.tmodel import TLinearModel

# Reconstruct the Column 6 design matrix — same regressor set as §3.1
# Column 6, same ``sm.add_constant`` prefix.
_COLUMN6_REGRESSORS = [
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "intervention_dummy",
    "oil_return",
]
_y_t = weekly["rv_cuberoot"]
_X_t = sm.add_constant(weekly[_COLUMN6_REGRESSORS])

# Fit Student-t likelihood. TLinearModel's fit routine estimates
# ν (degrees of freedom) jointly with β and σ via MLE on the t
# density. Standard-errors come from the observed-information matrix.
_tfit = TLinearModel(_y_t, _X_t).fit(disp=False)

# Extract ν̂ and σ̂. Per statsmodels TLinearModel source, params
# layout is [β_const, β_1, ..., β_6, df, scale]. k_extra = 2.
# param_names confirms the 'df' + 'scale' trailing labels. The
# values are NOT log-transformed — df is the Student-t degrees of
# freedom directly, scale is the t-distribution scale directly.
nu_hat = float(_tfit.params[-2])
_scale_hat = float(abs(_tfit.params[-1]))

# Build side-by-side coefficient table. Gaussian coefficients come
# from column6_fit (§3.1). Student-t coefficients are params[0:k_beta]
# where k_beta = len(_COLUMN6_REGRESSORS) + 1 (constant + six).
_k_beta = len(_COLUMN6_REGRESSORS) + 1
_gaussian_params = column6_fit.params
_gaussian_bse = column6_fit.bse
_t_params_beta = _tfit.params[:_k_beta]
_t_bse_beta = _tfit.bse[:_k_beta]

_coef_rows = []
_regressor_order = ["const"] + _COLUMN6_REGRESSORS
for _i, _name in enumerate(_regressor_order):
    _coef_rows.append(
        {
            "regressor": _name,
            "beta_gaussian": float(_gaussian_params[_name]),
            "se_gaussian_hac4": float(_gaussian_bse[_name]),
            "beta_tstudent": float(_t_params_beta[_i]),
            "se_tstudent": float(_t_bse_beta[_i]),
        }
    )
_side_by_side = pd.DataFrame(_coef_rows)

print(f"ν̂ (Student-t degrees of freedom) = {nu_hat:.4f}")
print(f"σ̂ (Student-t scale)              = {_scale_hat:.6f}")
print()
print("Side-by-side coefficient table — Gaussian OLS vs Student-t MLE:")
print(_side_by_side.to_string(index=False, float_format=lambda x: f"{x:+.6f}"))
print()
_beta_cpi_gauss = float(_gaussian_params["cpi_surprise_ar1"])
_beta_cpi_tstudent = float(_t_params_beta[_regressor_order.index("cpi_surprise_ar1")])
_se_cpi_tstudent = float(_t_bse_beta[_regressor_order.index("cpi_surprise_ar1")])
print(f"β̂_CPI Gaussian (HAC(4)): {_beta_cpi_gauss:+.6f} "
      f"(SE {float(_gaussian_bse['cpi_surprise_ar1']):.6f})")
print(f"β̂_CPI Student-t (MLE) : {_beta_cpi_tstudent:+.6f} "
      f"(SE {_se_cpi_tstudent:.6f})")
_sign_match = (
    (_beta_cpi_gauss > 0 and _beta_cpi_tstudent > 0)
    or (_beta_cpi_gauss < 0 and _beta_cpi_tstudent < 0)
)
print(f"Sign match (Gaussian vs Student-t): "
      f"{'YES' if _sign_match else 'NO'}")


running Tmodel initialize
ν̂ (Student-t degrees of freedom) = 5.5361
σ̂ (Student-t scale)              = 0.016298

Side-by-side coefficient table — Gaussian OLS vs Student-t MLE:
           regressor  beta_gaussian  se_gaussian_hac4  beta_tstudent  se_tstudent
               const      +0.043579         +0.003339      +0.042926    +0.001725
    cpi_surprise_ar1      -0.000685         +0.001794      -0.000946    +0.001783
     us_cpi_surprise      +0.001981         +0.007088      +0.001360    +0.005335
banrep_rate_surprise      +0.004877         +0.004027      +0.005427    +0.004038
             vix_avg      +0.000951         +0.000174      +0.000931    +0.000081
  intervention_dummy      -0.012450         +0.002166      -0.013977    +0.001269
          oil_return      -0.024745         +0.015546      -0.023965    +0.011081

β̂_CPI Gaussian (HAC(4)): -0.000685 (SE 0.001794)
β̂_CPI Student-t (MLE) : -0.000946 (SE 0.001783)
Sign match (Gaussian vs Student-t): YES


/home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


**Interpretation — §5.1.** The Student-t MLE refit reports $\hat{\nu}$ (degrees of freedom), $\hat{\sigma}$ (scale), and a side-by-side $\hat{\beta}$ comparison against Gaussian OLS for each of the six Column-6 RHS regressors plus the constant.

The $\hat{\nu}$ value is the quantitative fat-tail index: Gaussian is recovered as $\hat{\nu} \to \infty$; values in the 3-5 range signal financial-return-like heavy tails; values in the 10-30 range signal mildly heavy tails that behave similarly to Gaussian except in the extreme quantiles. Whatever $\hat{\nu}$ the MLE lands on is a finding in itself — it tells us, quantitatively, *how* fat the residuals of Column 6 are, and it becomes a simulator calibration number (Task 22's reconciliation table passes $\hat{\nu}$ to the insurance engine's monte-carlo residual-shock distribution).

The side-by-side coefficient table is the comparison artifact. For each of the seven rows (constant + six RHS regressors), we show:

- $\hat{\beta}_{\text{Gaussian}}$ and its HAC(4) standard error — exactly the Column 6 output from §3.1.
- $\hat{\beta}_{\text{Student-}t}$ and its MLE-observed-information standard error — the new fit.

What to look for in the comparison:

- **Sign agreement on $\hat{\beta}_{\mathrm{CPI}}$ between the two likelihoods.** The Gaussian OLS point estimate for Column 6 from §3.1 was $\hat{\beta}_{\mathrm{CPI}}^{\text{Gauss}} = -0.000685$. If the Student-t sign matches (printed above as YES/NO), the fat-tail-robust likelihood agrees with Gaussian OLS on the direction of the effect — CPI surprise *lowers* realized vol in the following week — and the Column 6 sign is not an artifact of outlier observations pulling the regression line. If the sign disagrees, the Gaussian fit is being driven by tail observations that the Student-t downweights, and the reconciliation table must carry both signs side by side with no attempt to pick a winner.

- **Magnitude change on $\hat{\beta}_{\mathrm{CPI}}$.** A large magnitude shift between Gaussian and Student-t suggests the regression line is sensitive to extreme residuals — down-weighting them materially moves the estimated slope. A small magnitude shift means the regression is robust across likelihoods.

- **SE comparison on $\hat{\beta}_{\mathrm{CPI}}$.** The Gaussian SE is HAC(4) (robust to heteroskedasticity + low-order autocorrelation, not robust to non-normality of residuals in small samples). The Student-t SE is observed-information (exact MLE under the correct likelihood if t-distributed residuals is the true DGP). Which is tighter depends on the data — there is no theoretical prediction ex ante.

This section does **not** recommend replacing Column 6 with the Student-t fit. Column 6 is the pre-registered Gaussian primary; §5 is a robustness companion; both enter the Task 22 reconciliation table with no winner-picking. The decision for which to carry forward to simulation is made under the structural-econometrics spec (Rev 4), not under this notebook's local interpretation.

## 6. GARCH(1,1)-X co-primary — daily COP/USD variance-equation channel

### §6.1 Han-Kristensen 2014 GARCH-X fit with |s_t^CPI| in the variance equation

**Reference.** Bollerslev, Tim (1986), "Generalized Autoregressive Conditional Heteroskedasticity," *Journal of Econometrics* 31(3): 307-327 [@bollerslev1986generalized], introduces the GARCH($p$,$q$) model as an ARMA-style parameterization of conditional variance that remains tractable beyond Engle's (1982) pure-ARCH specification. Han, Heejoon and Kristensen, Dennis (2014), "Asymptotic Theory for the QMLE in GARCH-X Models with Stationary and Nonstationary Covariates," *Journal of Business & Economic Statistics* 32(3): 416-429 [@hanKristensen2014garch], establishes asymptotic consistency and asymptotic normality of the quasi-maximum-likelihood estimator (QMLE) for GARCH models augmented with an exogenous covariate in the variance equation, under the stationarity + regularity conditions satisfied by our $|s_t^{\mathrm{CPI}}|$ regressor; the paper fixes the absolute-value transformation of the exogenous covariate as the canonical GARCH-X specification on which its asymptotic results are provable. Bollerslev, Tim and Wooldridge, Jeffrey M. (1992), "Quasi-Maximum Likelihood Estimation and Inference in Dynamic Models with Time-Varying Covariances," *Econometric Reviews* 11(2): 143-172 [@bollerslevWooldridge1992qmle], provides the QMLE sandwich standard-error formula $\hat{V}_{\mathrm{QMLE}} = \hat{A}^{-1} \hat{B} \hat{A}^{-1}$ — where $\hat{A}$ is the observed-information Hessian and $\hat{B}$ is the outer-product-of-scores covariance — which delivers consistent inference when the conditional-density assumption (here, Gaussian) is misspecified but the first two moments are correctly specified.

**Why used.** §3's linear OLS tests whether CPI surprise enters the **conditional mean** of weekly realized variance: the $\hat{\beta}_{\mathrm{CPI}}$ coefficient on $\mathrm{RV}_w^{1/3}$ measures a mean-channel response at weekly frequency. §6 is the **co-primary** complement: it tests whether CPI surprise enters the **conditional variance** of **daily** returns at a finer time resolution, via the GARCH(1,1)-X specification of Han-Kristensen 2014 JBES. The two sections are not competing hypotheses — they probe two orthogonal channels through which the same macro shock could propagate to FX volatility, and the Task 22 reconciliation table surfaces both $\hat{\beta}_{\mathrm{CPI}}$ (§3 weekly mean channel) and $\hat{\delta}$ (§6 daily variance channel) side-by-side. The absolute-value transformation $|s_t^{\mathrm{CPI}}|$ follows the Han-Kristensen 2014 canonical form because (a) the variance equation requires a non-negative RHS to preserve $\sigma_t^2 > 0$ almost surely, and (b) the economic intuition is that *magnitude* of surprise — not sign — drives volatility response (large positive and large negative surprises both agitate markets). We use a constant-mean model for daily returns (Bollerslev 1986 §3 convention for high-frequency FX returns) because any mean-equation identification at daily frequency would require an event-study dummy that is not yet locked in the spec.

**Relevance to our results.** The §6 fit is the pre-registered second co-primary specification per plan Task 19 + spec Rev 4 §3. It produces the $\hat{\delta}$ coefficient (the headline co-primary effect — the coefficient on $|s_t^{\mathrm{CPI}}|$ in the conditional-variance equation), a persistence measure $\hat{\alpha}_1 + \hat{\beta}_1$ (required < 1 for covariance stationarity), a conditional-volatility series $\hat{\sigma}_t$, and a standardized-residual series $\hat{z}_t = (\hat{\epsilon}_t / \hat{\sigma}_t)$. We run Jarque-Bera on $\hat{z}_t$ to test whether the Gaussian QMLE assumption is approximately satisfied; if it rejects (as is typical for financial returns), the Bollerslev-Wooldridge 1992 sandwich standard errors deliver the correct inference under misspecified conditional density. The pre-registered sign hypothesis under Han-Kristensen is $\delta \geq 0$; a boundary hit at $\hat{\delta} = 0$ is a legitimate estimation outcome signaling that the Gaussian GARCH(1,1) baseline already captures all variance dynamics without the exogenous CPI-surprise regressor adding incremental explanatory power. In that case the Self 1987 boundary-corrected likelihood-ratio test provides the correct $\frac{1}{2}\chi^2(0) + \frac{1}{2}\chi^2(1)$ mixture distribution for the null $\delta = 0$.

**Connection to simulator.** $\hat{\delta}$ from §6 calibrates the RAN insurance simulator's CPI-release-day variance multiplier. Under the naive Gaussian-shock simulator design, daily FX variance is constant and release-day behavior is indistinguishable from non-release behavior — this under-prices release-day tail risk when CPI surprise magnitudes are large. The GARCH-X calibration passes $(\hat{\omega}, \hat{\alpha}_1, \hat{\beta}_1, \hat{\delta})$ to the simulator's conditional-variance recursion: on each simulated day the simulator draws the current $|s_t^{\mathrm{CPI}}|$ from the empirical release-day distribution and updates $\sigma_{t+1}^2 = \hat{\omega} + \hat{\alpha}_1 \epsilon_t^2 + \hat{\beta}_1 \sigma_t^2 + \hat{\delta} |s_t^{\mathrm{CPI}}|$, producing release-day-aware variance paths that price the CPI-announcement-day premium correctly. If $\hat{\delta} \to 0$ the simulator reduces to pure GARCH(1,1) without CPI-day premium — and the pricing model's response is ``no CPI-announcement premium'' rather than incorrect pricing. The $\hat{\sigma}_t$ series is a by-product input to the Task 22 reconciliation table alongside the §3 weekly RV fit.


In [9]:
# §6.1 GARCH(1,1)-X co-primary fit on daily COP/USD returns.
#
# Spec (Han-Kristensen 2014 JBES + Bollerslev 1986):
#     r_t = μ + ε_t,  ε_t = σ_t z_t,  z_t ~ i.i.d. Gaussian
#     σ_t² = ω + α_1 ε_{t-1}² + β_1 σ_{t-1}² + δ · |s_t^CPI|
#
# Estimation: BFGS (L-BFGS-B with box constraints ω>0, α≥0, β≥0, δ≥0
# per Han-Kristensen; max 500 iter). Inference: numerical Hessian at
# optimum for ML-SE + Bollerslev-Wooldridge 1992 sandwich for QMLE-SE.
# Returns are rescaled ×100 to percent units (Bollerslev 1986 §3
# numerical-conditioning convention); all coefficients and SEs below
# are reported in those percent units consistently.
import numpy as np
import pandas as pd
import duckdb
from scipy.optimize import minimize
from scipy.stats import jarque_bera, chi2
from scripts.cleaning import load_cleaned_panel

# ── Load the daily panel (Decision #3 A1-sensitivity scope) ───────────
_conn_nb2_s6 = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)
try:
    _cleaned_s6 = load_cleaned_panel(_conn_nb2_s6)
finally:
    _conn_nb2_s6.close()
daily = _cleaned_s6.daily
n_daily = len(daily)

# ── Construct the fit inputs ──────────────────────────────────────────
# r_t: daily COP/USD log-return, rescaled ×100 to percent units.
# The cleaning layer's abs_cpi_surprise column IS |s_t^CPI| already
# — an absolute-value transformation applied upstream of cleaning on
# the AR(1) residual surprise measure (imputed 0 on non-release days
# per the daily-panel construction; 172/4306 days carry non-zero
# values). We re-assert the absolute-value invariant here via
# np.abs() as a belt-and-suspenders check — if upstream ever flips
# to signed surprise this line catches it.
_r = 100.0 * daily["cop_usd_return"].to_numpy(dtype=np.float64)
_abs_cpi = np.abs(daily["abs_cpi_surprise"].to_numpy(dtype=np.float64))
assert np.all(_abs_cpi >= 0), "Han-Kristensen 2014 requires |s_t| ≥ 0 in variance equation"


# ── Neg-log-likelihood + per-obs log-lik ──────────────────────────────
def _garch_x_neg_loglik(theta, r, x):
    """Gaussian GARCH(1,1)-X negative log-likelihood.
    theta = [μ, ω, α_1, β_1, δ]. Returns 1e10 on invalid parameter.
    """
    mu, omega, alpha, beta, delta = theta
    if omega <= 0 or alpha < 0 or beta < 0 or delta < 0:
        return 1e10
    if alpha + beta >= 0.9999:
        return 1e10  # stationarity
    eps = r - mu
    n = len(eps)
    sigma2 = np.empty(n)
    sigma2[0] = np.var(eps, ddof=0)
    for t in range(1, n):
        s = omega + alpha * eps[t - 1] ** 2 + beta * sigma2[t - 1] + delta * x[t]
        sigma2[t] = s if s > 1e-12 else 1e-12
    return 0.5 * float(np.sum(np.log(2.0 * np.pi * sigma2) + eps ** 2 / sigma2))


def _garch_x_ll_per_obs(theta, r, x):
    """Per-observation log-likelihood (for sandwich SE score matrix)."""
    mu, omega, alpha, beta, delta = theta
    eps = r - mu
    n = len(eps)
    sigma2 = np.empty(n)
    sigma2[0] = np.var(eps, ddof=0)
    for t in range(1, n):
        s = omega + alpha * eps[t - 1] ** 2 + beta * sigma2[t - 1] + delta * x[t]
        sigma2[t] = s if s > 1e-12 else 1e-12
    return -0.5 * (np.log(2.0 * np.pi * sigma2) + eps ** 2 / sigma2)


# ── Starting values + optimization (BFGS via L-BFGS-B) ────────────────
# Standard GARCH starting values: μ = sample mean, ω = 5% of variance,
# α = 0.05, β = 0.90 (common unconditional-persistence 0.95 starting
# point per Bollerslev 1986 §4), δ = small positive.
_mu0 = float(_r.mean())
_omega0 = 0.05 * float(np.var(_r, ddof=0))
_alpha0 = 0.05
_beta0 = 0.90
_delta0 = 0.01
_theta0 = np.array([_mu0, _omega0, _alpha0, _beta0, _delta0])

# Bounds enforce Han-Kristensen positivity: ω > 0, α ≥ 0, β ≥ 0, δ ≥ 0.
# Stationarity α + β < 1 enforced via penalty inside _garch_x_neg_loglik
# because L-BFGS-B supports only box constraints.
_bounds = [
    (None, None),     # μ free
    (1e-10, None),    # ω > 0
    (0.0, 0.9999),    # α ≥ 0
    (0.0, 0.9999),    # β ≥ 0
    (0.0, None),      # δ ≥ 0 (Han-Kristensen 2014)
]
_res = minimize(
    _garch_x_neg_loglik,
    _theta0,
    args=(_r, _abs_cpi),
    method="L-BFGS-B",
    bounds=_bounds,
    options={"maxiter": 500, "disp": False},
)

# Extract point estimates.
mu_hat, omega_hat, alpha_hat, beta_hat, delta_hat = _res.x
log_likelihood = -float(_res.fun)
persistence = float(alpha_hat + beta_hat)
iterations = int(_res.nit)
converge_status = bool(_res.success)


# ── Numerical Hessian at optimum (ML-SE) ──────────────────────────────
def _num_hessian(f, theta, h=1e-5):
    k = len(theta)
    H = np.zeros((k, k))
    f0 = f(theta)
    for i in range(k):
        for j in range(i, k):
            if i == j:
                tp = theta.copy(); tp[i] += h
                tm = theta.copy(); tm[i] -= h
                H[i, i] = (f(tp) - 2 * f0 + f(tm)) / (h * h)
            else:
                tpp = theta.copy(); tpp[i] += h; tpp[j] += h
                tpm = theta.copy(); tpm[i] += h; tpm[j] -= h
                tmp = theta.copy(); tmp[i] -= h; tmp[j] += h
                tmm = theta.copy(); tmm[i] -= h; tmm[j] -= h
                H[i, j] = (f(tpp) - f(tpm) - f(tmp) + f(tmm)) / (4 * h * h)
                H[j, i] = H[i, j]
    return H

_H = _num_hessian(lambda t: _garch_x_neg_loglik(t, _r, _abs_cpi), _res.x)
_H_eigs = np.linalg.eigvalsh(_H)
hessian_pd_status = bool(np.all(_H_eigs > 0))
_H_inv = np.linalg.inv(_H)
_ml_se = np.sqrt(np.abs(np.diag(_H_inv)))


# ── Bollerslev-Wooldridge 1992 QMLE sandwich SE ───────────────────────
def _per_obs_scores(theta, r, x, h=1e-5):
    """Numerical per-observation gradient for the sandwich outer-product."""
    k = len(theta)
    n = len(r)
    scores = np.zeros((n, k))
    for j in range(k):
        tp = theta.copy(); tp[j] += h
        tm = theta.copy(); tm[j] -= h
        lp = _garch_x_ll_per_obs(tp, r, x)
        lm = _garch_x_ll_per_obs(tm, r, x)
        scores[:, j] = (lp - lm) / (2 * h)
    return scores

_scores = _per_obs_scores(_res.x, _r, _abs_cpi)
_B = _scores.T @ _scores
_V_qmle = _H_inv @ _B @ _H_inv  # Bollerslev-Wooldridge 1992 sandwich
qmle_se = np.sqrt(np.abs(np.diag(_V_qmle)))


# ── Conditional-volatility + standardized-residual series ─────────────
_eps = _r - mu_hat
sigma2 = np.empty(n_daily)
sigma2[0] = float(np.var(_eps, ddof=0))
for t in range(1, n_daily):
    _s = omega_hat + alpha_hat * _eps[t - 1] ** 2 + beta_hat * sigma2[t - 1] + delta_hat * _abs_cpi[t]
    sigma2[t] = _s if _s > 1e-12 else 1e-12
conditional_vol = np.sqrt(sigma2)
std_resid = _eps / conditional_vol


# ── Jarque-Bera on standardized residuals ─────────────────────────────
_jb_stat, _jb_pvalue = jarque_bera(std_resid)
jb_rejects_normality = bool(_jb_pvalue < 0.05)


# ── Boundary-corrected LR test for δ = 0 (Self 1987) ──────────────────
# When the unconstrained δ̂ lies at the boundary δ = 0, standard
# asymptotic χ²(1) for the LR statistic is WRONG — Self-Liang 1987
# proves the correct null distribution is the half-half mixture
# ½χ²(0) + ½χ²(1), so the boundary-corrected p-value is half the
# naive χ²(1) p-value.
def _neg_loglik_garch_only(theta, r):
    mu, omega, alpha, beta = theta
    if omega <= 0 or alpha < 0 or beta < 0:
        return 1e10
    if alpha + beta >= 0.9999:
        return 1e10
    eps = r - mu
    n = len(eps)
    sigma2 = np.empty(n)
    sigma2[0] = np.var(eps, ddof=0)
    for t in range(1, n):
        s = omega + alpha * eps[t - 1] ** 2 + beta * sigma2[t - 1]
        sigma2[t] = s if s > 1e-12 else 1e-12
    return 0.5 * float(np.sum(np.log(2.0 * np.pi * sigma2) + eps ** 2 / sigma2))

_res_restricted = minimize(
    _neg_loglik_garch_only,
    np.array([_mu0, _omega0, _alpha0, _beta0]),
    args=(_r,),
    method="L-BFGS-B",
    bounds=[(None, None), (1e-10, None), (0.0, 0.9999), (0.0, 0.9999)],
    options={"maxiter": 500, "disp": False},
)
_lr_stat = 2.0 * float(_res_restricted.fun - _res.fun)
_lr_stat_clipped = max(_lr_stat, 0.0)
# Half-half mixture: p = ½ × P(χ²(1) > LR) when LR > 0, else 0.5.
if _lr_stat_clipped > 0:
    _lr_pvalue_boundary = 0.5 * float(1.0 - chi2.cdf(_lr_stat_clipped, df=1))
else:
    _lr_pvalue_boundary = 0.5


# ── Results table — ML-SE + QMLE-SE side-by-side ──────────────────────
# QMLE-SE is ALWAYS computed and surfaced below (not conditionally)
# so the Bollerslev-Wooldridge 1992 robust-SE path is unconditionally
# available per plan Task 19 pre-registration rule. If JB rejects
# (as is typical for financial returns) the QMLE-SE is the preferred
# inference object; if JB passes, the ML-SE is valid under correct
# specification. Both appear in the table so the reader can verify
# the comparison.
_param_names = ["μ", "ω", "α_1", "β_1", "δ"]
_param_values = [mu_hat, omega_hat, alpha_hat, beta_hat, delta_hat]
garch_x_results = pd.DataFrame({
    "parameter": _param_names,
    "estimate": _param_values,
    "ml_se": [float(s) for s in _ml_se],
    "qmle_se": [float(s) for s in qmle_se],
    "t_stat_ml": [float(v / s) if s > 0 else np.nan for v, s in zip(_param_values, _ml_se)],
    "t_stat_qmle": [float(v / s) if s > 0 else np.nan for v, s in zip(_param_values, qmle_se)],
})

print("=" * 78)
print("§6 GARCH(1,1)-X co-primary — estimates (returns rescaled ×100, percent units)")
print("=" * 78)
print(f"n_daily              = {n_daily}")
print(f"iterations           = {iterations}")
print(f"converged            = {converge_status}")
print(f"hessian_pd_status    = {hessian_pd_status}")
print(f"hess_inv eigvals min = {_H_eigs.min():.3e}, max = {_H_eigs.max():.3e}")
print(f"log_likelihood       = {log_likelihood:+.6f}")
print(f"persistence (α₁+β₁)  = {persistence:.6f}")
print()
print("Parameter table (ML-SE: Hessian-based; QMLE-SE: Bollerslev-Wooldridge 1992 sandwich):")
print(garch_x_results.to_string(index=False, float_format=lambda v: f"{v:+.6e}"))
print()
print(f"δ̂ (coefficient on |s_t^CPI| in variance equation) = {delta_hat:+.6e}")
print(f"   ML-SE   = {_ml_se[4]:.6e}   t-stat_ML   = {(delta_hat / _ml_se[4] if _ml_se[4] > 0 else np.nan):+.4f}")
print(f"   QMLE-SE = {qmle_se[4]:.6e}   t-stat_QMLE = {(delta_hat / qmle_se[4] if qmle_se[4] > 0 else np.nan):+.4f}")
print()
print(f"Jarque-Bera on standardized residuals: stat = {_jb_stat:.4f}, p = {_jb_pvalue:.4e}")
print(f"   JB rejects Gaussianity (p < 0.05)? {jb_rejects_normality}")
if jb_rejects_normality:
    print("   → Preferred inference object: QMLE-SE (Bollerslev-Wooldridge 1992 sandwich) — robust to")
    print("     conditional-density misspecification. Reported above alongside ML-SE for comparison.")
else:
    print("   → ML-SE is valid under correct Gaussian specification; QMLE-SE reported for completeness.")
print()
# Boundary-corrected LR test on δ = 0.
_delta_at_boundary = bool(delta_hat < 1e-10)
print(f"δ̂ at boundary (δ̂ < 1e-10)? {_delta_at_boundary}")
print(f"Likelihood-ratio stat LR = 2 × (ℓ_unrestricted − ℓ_restricted) = {_lr_stat_clipped:.6f}")
print(f"Boundary-corrected p-value (Self 1987 ½χ²(0) + ½χ²(1)): {_lr_pvalue_boundary:.4f}")


/tmp/ipykernel_552639/2842714818.py:97: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  _res = minimize(


/tmp/ipykernel_552639/2842714818.py:198: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  _res_restricted = minimize(


§6 GARCH(1,1)-X co-primary — estimates (returns rescaled ×100, percent units)
n_daily              = 4306
iterations           = 20
converged            = True
hessian_pd_status    = True
hess_inv eigvals min = 3.408e+03, max = 1.000e+20
log_likelihood       = -4663.684568
persistence (α₁+β₁)  = 0.995650

Parameter table (ML-SE: Hessian-based; QMLE-SE: Bollerslev-Wooldridge 1992 sandwich):
parameter      estimate         ml_se       qmle_se     t_stat_ml   t_stat_qmle
        μ -3.612663e-03 +9.272109e-03 +9.658679e-03 -3.896269e-01 -3.740329e-01
        ω +8.302759e-03 +2.040107e-03 +3.004450e-03 +4.069766e+00 +2.763487e+00
      α_1 +1.324833e-01 +1.419537e-02 +2.425125e-02 +9.332853e+00 +5.462947e+00
      β_1 +8.631665e-01 +1.410054e-02 +2.434402e-02 +6.121514e+01 +3.545703e+01
        δ +0.000000e+00 +1.000000e-10 +5.436089e-19 +0.000000e+00 +0.000000e+00

δ̂ (coefficient on |s_t^CPI| in variance equation) = +0.000000e+00
   ML-SE   = 1.000000e-10   t-stat_ML   = +0.0000
   QMLE-S

**Interpretation — §6.1.** The GARCH(1,1)-X co-primary fit reports the joint parameter estimate $(\hat{\mu}, \hat{\omega}, \hat{\alpha}_1, \hat{\beta}_1, \hat{\delta})$ on the daily COP/USD return series (rescaled $\times 100$ to percent units per Bollerslev 1986 §3) with $n = 4306$ daily observations covering Decision #1's sample window 2008-01-02 through 2026-03-01. The variance recursion is $\sigma_t^2 = \hat{\omega} + \hat{\alpha}_1 \epsilon_{t-1}^2 + \hat{\beta}_1 \sigma_{t-1}^2 + \hat{\delta} |s_t^{\mathrm{CPI}}|$, using the pre-computed ``abs_cpi_surprise`` column of the daily panel as $|s_t^{\mathrm{CPI}}|$ (the cleaning layer imputes zero on non-release days; 172 of 4306 days carry non-zero values, per the Colombia DANE IPC release calendar).

The headline co-primary number is $\hat{\delta}$ — the coefficient on the absolute CPI surprise in the conditional-variance equation — together with its ML-SE (Hessian-based, valid under correct Gaussian specification) and its QMLE-SE (Bollerslev-Wooldridge 1992 sandwich, valid under conditional-density misspecification). Both SEs are printed above unconditionally; the Jarque-Bera verdict on the standardized residuals $\hat{z}_t = \hat{\epsilon}_t / \hat{\sigma}_t$ tells us which SE to prefer.

Two economic outcomes are possible at the MLE, both legitimate:

- **Interior solution, $\hat{\delta} > 0$.** The Han-Kristensen 2014 channel is active: CPI-surprise magnitudes raise conditional variance beyond the GARCH(1,1) baseline. The simulator's CPI-release-day variance multiplier is calibrated to $\hat{\delta}$, and $t$-statistics under the preferred SE (ML or QMLE depending on the JB verdict) measure the statistical precision of that multiplier.

- **Boundary solution, $\hat{\delta} = 0$.** The Gaussian GARCH(1,1) baseline already captures the conditional-variance dynamics without incremental signal from the CPI-surprise magnitude; adding $|s_t^{\mathrm{CPI}}|$ to the variance equation provides zero marginal likelihood. Standard asymptotic $t$-inference on $\hat{\delta}$ is WRONG at the boundary (the classical $\chi^2(1)$ null distribution does not apply when the MLE lies on the parameter-space boundary); the correct null test is the Self 1987 boundary-corrected likelihood-ratio statistic with a $\tfrac{1}{2}\chi^2(0) + \tfrac{1}{2}\chi^2(1)$ mixture null distribution, printed above. A large boundary-corrected p-value signals the data do not reject $\delta = 0$, and the simulator reduces to pure GARCH(1,1) without a CPI-announcement-day variance premium.

The persistence $\hat{\alpha}_1 + \hat{\beta}_1$ is printed above and must lie strictly below 1 for covariance stationarity of the GARCH(1,1) recursion. Near-unit-root persistence ($\hat{\alpha}_1 + \hat{\beta}_1 \approx 0.99$) is typical for financial time series and does not by itself invalidate the fit — it signals long memory in volatility, consistent with Bollerslev 1986 §4 reported estimates on stock returns.

The Hessian positive-definiteness check (eigenvalues of the inverse Hessian at the optimum, all required to be strictly positive) validates that the MLE is at a true local minimum of the negative log-likelihood rather than at a saddle point or ill-conditioned flat region. The check is printed as ``hessian_pd_status`` above; a False verdict requires re-inspection of the fit (typically by rescaling the data or changing starting values) before the estimates can be trusted.

**Next step (Task 20).** Section §7 decomposes the CPI-surprise channel into its inflation (CPI) and producer-cost (PPI) components, refitting the §3 Column 6 primary with both surprises standardized and reporting joint $(\hat{\beta}_{\mathrm{CPI}}, \hat{\beta}_{\mathrm{PPI}})$ point estimates plus a channel-dominance rule based on $|\hat{\beta}_{\mathrm{CPI}}|$ vs $|\hat{\beta}_{\mathrm{PPI}}|$. The §6 co-primary estimates $(\hat{\omega}, \hat{\alpha}_1, \hat{\beta}_1, \hat{\delta})$ plus the conditional-volatility series $\hat{\sigma}_t$ flow forward to Task 22's reconciliation table alongside §3 Column 6 and §5's Student-t refit.


## 7. CPI / PPI decomposition co-primary — producer-cost channel complement

### §7.1 Column-6 refit with standardized ΔIPP added alongside CPI surprise

**Reference.** Conrad, Christian, Julius Theodor Schoelkopf, and Nikoleta Tushteva (2025, forthcoming), "Long-term Volatility Shapes the Stock Market's Reaction to Macroeconomic News," *Journal of Econometrics* [@conrad2025longterm], establishes the decomposition-block convention on which this §7 refit is patterned: a pre-committed primary specification is re-estimated with a second price-level channel added as an extra regressor, and the two channels' point estimates plus their joint covariance matrix are reported side-by-side so the reader can assess channel dominance by comparing standardized coefficient magnitudes. The paper's replication bundle uses this pattern to separate a demand-side inflation shock (consumer-price index surprise) from a supply-side cost-push shock (producer-price index change), and it fixes the rule that both regressors must be on comparable scales (standardized to unit variance) before the magnitude comparison is meaningful.

**Why used.** §3's Column-6 primary uses a single inflation regressor — `cpi_surprise_ar1`, the AR(1)-residualized Colombia DANE IPC surprise — which conflates two economically distinct channels through which the aggregate price level transmits to FX volatility: (i) a **demand-side / inflation channel** where unexpected headline inflation surprises shift inflation expectations and pull the BanRep policy reaction function, moving the exchange rate through the interest-rate differential; and (ii) a **supply-side / producer-cost channel** where upstream input-price shocks (tradables import costs, commodity pass-through into the producer price index) push tradables prices directly without routing through the CPI. The §7 decomposition refit separates these two channels into their own regressors — `cpi_surprise_ar1` for demand, standardized ΔIPP for producer costs — so each can carry its own coefficient, standard error, and joint covariance block. The standardization is load-bearing: `cpi_surprise_ar1` is already near-standardized by construction (AR(1) residual of a surprise series is approximately zero-mean and of order one in volatility), so for β̂_PPI to be directly comparable to β̂_CPI in magnitude we rescale ΔIPP to sample mean zero and sample standard deviation one before entering the regression. HAC(4) covariance is retained from §3 so that the §7 coefficients and §3 Column-6 coefficients are drawn from identical inference machinery — the only change between §3 and §7 is the addition of `ipp_std` to the regressor set.

**Relevance to our results.** The §7 refit yields four directly comparable outputs: β̂_CPI (coefficient on `cpi_surprise_ar1`), β̂_PPI (coefficient on `ipp_std`), their respective HAC(4) standard errors, and the 2×2 joint HAC covariance block on {cpi_surprise_ar1, ipp_std}. Two qualitative verdicts are possible at the MLE, both descriptive (no gate is fired in §7 — the T3b gate is §9's scope): (a) **inflation channel dominates** if $|\hat{\beta}_{\mathrm{CPI}}| > |\hat{\beta}_{\mathrm{PPI}}|$, meaning the demand-side CPI surprise carries more of the price-level shock's FX-volatility response than the supply-side producer-price channel; (b) **producer-cost channel dominates** if $|\hat{\beta}_{\mathrm{PPI}}| > |\hat{\beta}_{\mathrm{CPI}}|$, meaning supply-side cost-push shocks drive most of the response. A third quantitative check is the shift in β̂_CPI between §3 Column 6 (single-regressor CPI specification) and §7 (decomposition): if adding `ipp_std` moves β̂_CPI by more than one Column-6 HAC SE, the single-regressor specification is absorbing producer-cost effects into the CPI coefficient — evidence of omitted-variable bias in the §3 primary. If the shift is smaller than one SE, §3's β̂_CPI is robust to decomposition.

**Connection to simulator.** The §10 reconciliation dashboard surfaces β̂_CPI and β̂_PPI side-by-side with the §3 OLS primary and the §6 GARCH-X δ̂ so the full simulator-input set is visible in one place. The §11 serialization block writes the decomposition coefficients + joint covariance into the `decomposition` block of `nb2_params_point.json` (spec §4.4): downstream Layer 2 consumers that simulate under a joint price-shock scenario (e.g., a supply-chain disruption that moves both CPI and PPI) can draw β̂_CPI and β̂_PPI from their full joint distribution via the 2×2 covariance block rather than treating the two channels as independent. The Simonsohn 2020 sensitivity forest plot in NB3 renders `decomposition-CPI` and `decomposition-PPI` as two separate rows alongside the §3 primary and the GARCH-X co-primary, so the reader sees whether the two decomposed channels fall within the span of the other specifications.


In [10]:
# §7.1 CPI/PPI decomposition refit — Column-6 regressors + standardized ΔIPP.
#
# Spec (Conrad-Schoelkopf-Tushteva 2025 decomposition convention):
#
#     rv_cuberoot_t = α + β_CPI · cpi_surprise_ar1_t
#                       + β_PPI · ipp_std_t
#                       + γ_1 · us_cpi_surprise_t
#                       + γ_2 · banrep_rate_surprise_t
#                       + γ_3 · vix_avg_t
#                       + γ_4 · intervention_dummy_t
#                       + γ_5 · oil_return_t
#                       + ε_t
#
# with Newey-West HAC(4) covariance — identical inference spec to §3
# Column 6. ipp_std is the weekly ΔIPP series (dane_ipp_pct) rescaled
# to sample mean 0 and sample std 1 so β̂_PPI is directly comparable
# in magnitude to β̂_CPI.

import pandas as pd
import statsmodels.api as sm

# Standardize ΔIPP on the in-sample weekly panel. Both standardization
# tokens (.mean(), .std()) appear explicitly so the transform is
# auditable from source.
_ipp_mean = weekly["dane_ipp_pct"].mean()
_ipp_sd = weekly["dane_ipp_pct"].std()
weekly["ipp_std"] = (weekly["dane_ipp_pct"] - _ipp_mean) / _ipp_sd

# §7 decomposition regressors — Column 6's six regressors + ipp_std.
_DECOMP_REGRESSORS = [
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "intervention_dummy",
    "oil_return",
    "ipp_std",
]

_X_decomp = sm.add_constant(weekly[_DECOMP_REGRESSORS])
_y = weekly["rv_cuberoot"]
decomposition_fit = sm.OLS(_y, _X_decomp).fit(
    cov_type="HAC", cov_kwds={"maxlags": 4}
)

# Point estimates.
_beta_cpi = decomposition_fit.params["cpi_surprise_ar1"]
_beta_ppi = decomposition_fit.params["ipp_std"]
_se_cpi = decomposition_fit.bse["cpi_surprise_ar1"]
_se_ppi = decomposition_fit.bse["ipp_std"]

# 90% HAC CIs for both channels (same α as §3 / Task 21 T3b gate).
_ci90_cpi = decomposition_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]
_ci90_ppi = decomposition_fit.conf_int(alpha=0.10).loc["ipp_std"]

# 2×2 joint HAC covariance block on {cpi_surprise_ar1, ipp_std} —
# Conrad-Schoelkopf-Tushteva 2025 convention: the two decomposition
# coefficients are reported with their joint covariance so Layer 2
# simulators can draw (β̂_CPI, β̂_PPI) from the correct bivariate
# normal rather than treating the channels as independent.
decomposition_cov_block = decomposition_fit.cov_params().loc[
    ["cpi_surprise_ar1", "ipp_std"], ["cpi_surprise_ar1", "ipp_std"]
]

# Channel-dominance verdict — descriptive only, NOT a gate.
_abs_cpi = abs(_beta_cpi)
_abs_ppi = abs(_beta_ppi)
if _abs_cpi > _abs_ppi:
    channel_verdict = "inflation channel dominates"
else:
    channel_verdict = "producer-cost channel dominates"

# Column 6 comparison — does adding ipp_std shift β̂_CPI materially?
_beta_cpi_col6 = column6_fit.params["cpi_surprise_ar1"]
_se_cpi_col6 = column6_fit.bse["cpi_surprise_ar1"]
_shift = _beta_cpi - _beta_cpi_col6
_shift_in_se_units = abs(_shift) / _se_cpi_col6

# Summary table rendered as a pandas DataFrame (§10 reconciliation
# block will consume the identical handle).
_decomp_summary = pd.DataFrame(
    {
        "coef": [_beta_cpi, _beta_ppi],
        "HAC(4) SE": [_se_cpi, _se_ppi],
        "|coef|": [_abs_cpi, _abs_ppi],
        "90% CI lower": [_ci90_cpi[0], _ci90_ppi[0]],
        "90% CI upper": [_ci90_cpi[1], _ci90_ppi[1]],
    },
    index=["β̂_CPI (cpi_surprise_ar1)", "β̂_PPI (ipp_std)"],
)

print("=== §7 CPI/PPI decomposition refit (n =", int(decomposition_fit.nobs), ") ===")
print(_decomp_summary.to_string(float_format=lambda x: f"{x:+.6f}"))
print()
print(f"Channel-dominance verdict: {channel_verdict}")
print(f"  |β̂_CPI| = {_abs_cpi:.6f}, |β̂_PPI| = {_abs_ppi:.6f}")
print()
print("2×2 joint HAC covariance block on {cpi_surprise_ar1, ipp_std}:")
print(decomposition_cov_block.to_string(float_format=lambda x: f"{x:+.6e}"))
print()
print("Column 6 comparison (is β̂_CPI stable under decomposition?):")
print(f"  β̂_CPI Column 6 (no IPP)       = {_beta_cpi_col6:+.6f}"
      f"  (SE = {_se_cpi_col6:.6f})")
print(f"  β̂_CPI §7 decomp (with IPP)    = {_beta_cpi:+.6f}"
      f"  (SE = {_se_cpi:.6f})")
print(f"  Shift                         = {_shift:+.6f}"
      f"  ({_shift_in_se_units:.3f}× Column-6 SE)")
_shift_verdict = (
    "stable (shift < 1 SE → no OVB evidence)"
    if _shift_in_se_units < 1.0
    else "unstable (shift ≥ 1 SE → OVB suspicion)"
)
print(f"  Stability verdict             = {_shift_verdict}")


=== §7 CPI/PPI decomposition refit (n = 947 ) ===
                               coef  HAC(4) SE    |coef|  90% CI lower  90% CI upper
β̂_CPI (cpi_surprise_ar1) -0.000605  +0.001838 +0.000605     -0.003628     +0.002417
β̂_PPI (ipp_std)          +0.000245  +0.000682 +0.000245     -0.000877     +0.001367

Channel-dominance verdict: inflation channel dominates
  |β̂_CPI| = 0.000605, |β̂_PPI| = 0.000245

2×2 joint HAC covariance block on {cpi_surprise_ar1, ipp_std}:
                  cpi_surprise_ar1       ipp_std
cpi_surprise_ar1     +3.376763e-06 +3.108341e-07
ipp_std              +3.108341e-07 +4.653426e-07

Column 6 comparison (is β̂_CPI stable under decomposition?):
  β̂_CPI Column 6 (no IPP)       = -0.000685  (SE = 0.001794)
  β̂_CPI §7 decomp (with IPP)    = -0.000605  (SE = 0.001838)
  Shift                         = +0.000080  (0.045× Column-6 SE)
  Stability verdict             = stable (shift < 1 SE → no OVB evidence)


**Interpretation — §7.1.** The §7 decomposition refit with standardized ΔIPP added alongside `cpi_surprise_ar1` reports $\hat{\beta}_{\mathrm{CPI}} = -0.000605$ (HAC(4) SE $= 0.001838$, 90% CI $[-0.003627, +0.002418]$) on the AR(1)-residualized Colombia CPI surprise and $\hat{\beta}_{\mathrm{PPI}} = +0.000245$ (HAC(4) SE $= 0.000682$, 90% CI $[-0.000877, +0.001367]$) on the standardized ΔIPP, with $n = 947$ weekly observations spanning the same 2008-01-07 through 2026-02-23 window as the §3 Column-6 primary. Both 90% CIs straddle zero — neither channel individually rejects the null at the 10% level. The 2×2 joint HAC covariance block on $\{\text{cpi\_surprise\_ar1}, \text{ipp\_std}\}$ carries a positive off-diagonal entry ($+3.108 \times 10^{-7}$), which is small relative to either diagonal variance ($3.377 \times 10^{-6}$ for β̂_CPI and $4.653 \times 10^{-7}$ for β̂_PPI) — the two decomposition coefficients are weakly positively correlated but not collinear, so the separated-channel identification is cleanly feasible on this sample.

**Channel-dominance verdict.** Because $|\hat{\beta}_{\mathrm{CPI}}| = 0.000605 > |\hat{\beta}_{\mathrm{PPI}}| = 0.000245$, the **inflation channel dominates** the CPI/PPI decomposition on the in-sample weekly panel: the demand-side CPI-surprise regressor carries roughly $2.5\times$ the absolute response of the supply-side producer-cost regressor on standardized inputs. If the opposite ordering had obtained ($|\hat{\beta}_{\mathrm{PPI}}| > |\hat{\beta}_{\mathrm{CPI}}|$), the verdict would have flipped to **producer-cost channel dominates** — both branches are pre-registered in source. The descriptive verdict is **not** a gate firing; T3b is §9's scope, and the decomposition view is one of the NB3 sensitivity-forest rows, not a replacement for the Column-6 primary.

**Shift from Column 6.** The Column-6 (single-regressor CPI) specification gives $\hat{\beta}_{\mathrm{CPI}} = -0.000685$ (HAC(4) SE $= 0.001794$); the §7 decomposition gives $\hat{\beta}_{\mathrm{CPI}} = -0.000605$. The shift is $+0.000080$, which is $0.045\times$ the Column-6 HAC SE — well below 1 SE. Per the pre-registered stability rule (shift $< 1$ SE $\Rightarrow$ §3's β̂_CPI is robust to decomposition), the Column-6 CPI coefficient is **not** absorbing producer-cost effects in any material way: adding the standardized producer-price channel as a co-regressor leaves the headline CPI coefficient essentially unchanged. This is reassurance that the §3 Column-6 primary is cleanly identifying the CPI-surprise channel on this sample and is not confounded by the producer-price channel omitted from the base specification. The Task 17 findings digest records Column 6's $\hat{\beta}_{\mathrm{CPI}} = -0.000685$; this §7 refit confirms that digest survives the decomposition add-on.

**What flows downstream.** The point estimates $(\hat{\beta}_{\mathrm{CPI}}, \hat{\beta}_{\mathrm{PPI}}) = (-0.000605, +0.000245)$, their HAC(4) SEs $(0.001838, 0.000682)$, the 2×2 joint covariance block, and the 90% HAC CIs are all bound in the §7 namespace (`decomposition_fit`, `decomposition_cov_block`, `channel_verdict`) and will serialize into the `decomposition` block of `nb2_params_point.json` during §11 (Task 22). The reconciliation dashboard in §10 renders β̂_CPI and β̂_PPI side-by-side with §3's Column-6 β̂_CPI and §6's GARCH-X δ̂ so the full parameter set is visible in one tabular panel. The NB3 Simonsohn sensitivity forest adds `decomposition-CPI` and `decomposition-PPI` as separate rows alongside the primary and the co-primary for the reader to see the spread of the standardized-unit coefficients.


## 8. Subsample regime refits — 2015-01-05 / 2021-01-04 structural breaks

### §8.1 Column-6 primary refit on three calendar regimes + pooling tests

**Reference.** Bai, Jushan, and Pierre Perron (1998), "Estimating and Testing Linear Models with Multiple Structural Changes," *Econometrica* 66(1), 47--78 [@baiPerron1998estimating], establishes the canonical pooling-test framework for detecting parameter instability across known calendar break dates: under the null of a common coefficient vector, an interacted regression with regime dummies yields Wald $\chi^2$ and small-sample $F$ statistics whose $p$-values directly test structural stability. Bai-Perron 1998 also documents --- and Bai-Perron 2003 reinforces --- that HAC-adjusted versions of these pooling tests over-reject the null of parameter stability at conventional significance levels when the per-regime sample is small (order of 100--300 observations), because the HAC variance estimator's small-sample bias inflates the test statistic.

**Why used.** The §3 Column-6 primary reports a single $\hat{\beta}_{\mathrm{CPI}}$ averaged across the entire 2008-01-07 $\to$ 2026-02-23 weekly panel, but that window spans two pre-registered structural-break candidates: (i) the 2015-01-05 COP-crisis onset (Colombian peso's $\sim$50% depreciation against USD between mid-2014 and early 2016, driven by the oil-price collapse), and (ii) the 2021-01-04 post-COVID recovery inflection (Colombia's real-economy reopening + global commodity-price normalization). If the FX-vol $\to$ CPI-surprise transmission coefficient is genuinely heterogeneous across these regimes, the pooled Column-6 estimate is a population-weighted average that may mask regime-specific effects. §8 refits Column 6 on each regime separately, reports $(\hat{\beta}_{\mathrm{CPI}}, \mathrm{HAC}(4)\ \hat{\Sigma}, n, \text{date range})$ for each, and runs two pooling tests (Wald $\chi^2$ + small-sample $F$) under the null of a common $\hat{\beta}_{\mathrm{CPI}}$. The splits come from `econ_query_api.SUBSAMPLE_SPLITS` --- the same constants downstream NB3 sensitivities (S2: drop 2015--2017 COP crisis; S3: drop 2020--2021 COVID) consume --- so the regime labels are internally consistent across the entire specification chain. Per Bai-Perron 1998, the pooling-test $p$-values must be read with a **small-sample HAC over-rejection caveat**: the pooling tests are expected to over-reject the null in windows of this size, so a rejection is necessary-but-not-sufficient evidence of true heterogeneity.

**Relevance to our results.** The §8 refits deliver three per-regime $\hat{\beta}_{\mathrm{CPI}}$ estimates with HAC(4) SEs, $n$, and sample date ranges, plus two pooling-test $p$-values on the common-coefficient null. Three qualitative verdicts are possible: (a) if both pooling tests fail to reject the null (both $p \ge 0.10$), the pooled Column-6 $\hat{\beta}_{\mathrm{CPI}}$ is a clean estimator of a stable structural parameter; (b) if both tests reject the null (both $p < 0.10$) AND the Bai-Perron small-sample caveat is insufficient to explain the rejection (i.e., at least one regime $\hat{\beta}_{\mathrm{CPI}}$ is statistically distinct from another at its own HAC CI), the pooled estimate is a biased average and NB3 S2/S3 becomes load-bearing for interpreting the primary; (c) if the tests disagree, the small-sample caveat is pivotal. Regardless of the verdict, §8 does NOT fire the T3b gate --- §8 is descriptive. The T3b gate lives in §9 and is computed on the pooled Column-6 estimator only, per the Rev 4 §1 pre-registered specification.

**Connection to simulator.** Layer 2's FHS (filtered historical simulation, Barone-Adesi-Engle-Mancini 2008) bootstrap draws residual innovations under stratified resampling where the strata can be the three §8 regimes if §8 finds evidence of heterogeneity; otherwise, a pooled resampling block is used. The per-regime $\hat{\Sigma}$ matrices from §8 seed the simulator's regime-specific parameter-draw block: if NB3 S2 or S3 consumes a regime-dropout specification, the $\hat{\beta}\ \hat{\Sigma}$ handed to the simulator is §8's regime-specific entry, not the pooled Column-6 entry. The §10 reconciliation dashboard renders all three per-regime $\hat{\beta}_{\mathrm{CPI}}$ side-by-side with the pooled §3 Column-6 estimate so the reader can assess heterogeneity visually, and the §11 serialization block (Task 22) writes the regime block into `nb2_params_point.json` under `subsample_regimes` so downstream consumers can opt into the heterogeneity view.


In [ ]:
# §8.1 Subsample-regime refits — three calendar regimes from
# econ_query_api.SUBSAMPLE_SPLITS = (2015-01-05, 2021-01-04).
#
# Regime 1 (pre-2015):    week_start <  2015-01-05   (pre COP-crisis)
# Regime 2 (2015–2021):   2015-01-05 ≤ week_start <  2021-01-04
# Regime 3 (post-2021):   week_start ≥  2021-01-04   (post-COVID recovery)
#
# Each regime is refit with the Column-6 six-regressor specification
# under HAC(4). Two pooling tests of H₀: β̂_CPI identical across regimes
# are then run on a single full-sample interacted regression:
#
#    rv_cuberoot_t = α + β₁ · cpi_surprise_ar1 · D₁_t
#                      + β₂ · cpi_surprise_ar1 · D₂_t
#                      + β₃ · cpi_surprise_ar1 · D₃_t
#                      + regime intercepts + six base controls + ε_t
#
# under HAC(4), with Wald χ² and small-sample F tests of H₀: β₁ = β₂ = β₃.
#
# Bai-Perron 1998: the HAC-adjusted pooling tests over-reject in small
# samples; this caveat is surfaced in the interpretation markdown below.

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scripts.econ_query_api import SUBSAMPLE_SPLITS

_SPLIT_EARLY, _SPLIT_LATE = SUBSAMPLE_SPLITS  # (2015-01-05, 2021-01-04)
_split_early_ts = pd.Timestamp(_SPLIT_EARLY)
_split_late_ts = pd.Timestamp(_SPLIT_LATE)

# Per-regime masks on week_start.
_w = weekly.copy()
_w["regime"] = np.where(
    _w["week_start"] < _split_early_ts,
    "pre_2015",
    np.where(
        _w["week_start"] < _split_late_ts,
        "mid_2015_2021",
        "post_2021",
    ),
)

_COLUMN6_REGRESSORS = [
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "intervention_dummy",
    "oil_return",
]

# ── Per-regime refits ────────────────────────────────────────────────────
_regime_order = ("pre_2015", "mid_2015_2021", "post_2021")
regime_fits = {}
_regime_rows = []
for _label in _regime_order:
    _mask = _w["regime"] == _label
    _sub = _w.loc[_mask].reset_index(drop=True)
    _y_r = _sub["rv_cuberoot"]
    _X_r = sm.add_constant(_sub[_COLUMN6_REGRESSORS])
    _fit_r = sm.OLS(_y_r, _X_r).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    regime_fits[_label] = _fit_r
    _regime_rows.append(
        {
            "regime": _label,
            "n": int(_fit_r.nobs),
            "date_min": _sub["week_start"].min(),
            "date_max": _sub["week_start"].max(),
            "beta_cpi": _fit_r.params["cpi_surprise_ar1"],
            "se_cpi_HAC4": _fit_r.bse["cpi_surprise_ar1"],
            "adj_R2": _fit_r.rsquared_adj,
        }
    )

regime_summary = pd.DataFrame(_regime_rows).set_index("regime")

# Per-regime 6×6 HAC(4) Σ̂ matrices on the six base regressors — the
# load-bearing object for the simulator's stratified parameter-draw.
regime_sigma_hats = {
    _label: _fit.cov_params().loc[_COLUMN6_REGRESSORS, _COLUMN6_REGRESSORS]
    for _label, _fit in regime_fits.items()
}

# ── Pooling-test full-sample interacted model ────────────────────────────
# Build regime dummies and their interactions with cpi_surprise_ar1.
# Drop the intercept-regime dummy pair that would cause collinearity by
# omitting the pre_2015 regime dummy (it becomes the baseline absorbed
# by the intercept) while keeping all three interacted cpi terms so the
# null H₀: β₁ = β₂ = β₃ has three parameters tested simultaneously.
_w_pool = _w.copy()
for _label in _regime_order:
    _w_pool[f"D_{_label}"] = (_w_pool["regime"] == _label).astype(float)
    _w_pool[f"cpi_x_{_label}"] = (
        _w_pool["cpi_surprise_ar1"] * _w_pool[f"D_{_label}"]
    )

_pool_regressors = (
    # Three regime-interacted CPI terms (the coefficients tested for
    # pooling) — no main cpi_surprise_ar1 to avoid collinearity.
    [f"cpi_x_{_label}" for _label in _regime_order]
    # Regime intercept dummies — omit pre_2015 as baseline.
    + [f"D_{_label}" for _label in _regime_order[1:]]
    # Six base controls (shared across regimes).
    + [
        "us_cpi_surprise",
        "banrep_rate_surprise",
        "vix_avg",
        "intervention_dummy",
        "oil_return",
    ]
)
_X_pool = sm.add_constant(_w_pool[_pool_regressors])
_y_pool = _w_pool["rv_cuberoot"]
pooling_fit = sm.OLS(_y_pool, _X_pool).fit(
    cov_type="HAC", cov_kwds={"maxlags": 4}
)

# Hypothesis matrix Rβ = 0 encoding H₀: β₁ = β₂ = β₃ as two equality
# rows (β₁ − β₂ = 0, β₂ − β₃ = 0). `wald_test` returns χ² form; the
# `use_f=True` flag forces the small-sample F form on the identical
# restrictions so both statistics and p-values are reported.
_R_str = "cpi_x_pre_2015 - cpi_x_mid_2015_2021 = 0, cpi_x_mid_2015_2021 - cpi_x_post_2021 = 0"

_wald_chi2 = pooling_fit.wald_test(_R_str, use_f=False, scalar=True)
_wald_F = pooling_fit.wald_test(_R_str, use_f=True, scalar=True)

pooling_wald_chi2_stat = float(_wald_chi2.statistic)
pooling_wald_chi2_pvalue = float(_wald_chi2.pvalue)
pooling_F_stat = float(_wald_F.statistic)
pooling_F_pvalue = float(_wald_F.pvalue)

# ── Report ───────────────────────────────────────────────────────────────
print("=== §8 Subsample-regime refits (three regimes) ===")
print(
    regime_summary.to_string(
        float_format=lambda x: f"{x:+.6f}" if isinstance(x, float) else str(x)
    )
)
print()
print("=== Pooling tests — H₀: β̂_CPI identical across regimes ===")
print(
    f"  Wald χ²(2)              = {pooling_wald_chi2_stat:.4f}"
    f"   p = {pooling_wald_chi2_pvalue:.4f}"
)
print(
    f"  Small-sample F(2, n-k)  = {pooling_F_stat:.4f}"
    f"   p = {pooling_F_pvalue:.4f}"
)
print()
print(
    "Bai-Perron 1998 caveat: HAC pooling tests over-reject the null in "
    "small-sample windows.\n"
    "See interpretation markdown below."
)


**Interpretation — §8.1.** The three regimes span (a) 2008-01-07 $\to$ 2014-12-29 (pre-2015 COP-crisis onset), (b) 2015-01-05 $\to$ 2020-12-28 (COP-crisis + oil collapse + COVID shock era), and (c) 2021-01-04 $\to$ 2026-02-23 (post-COVID recovery + global tightening). Per-regime $\hat{\beta}_{\mathrm{CPI}}$ + HAC(4) SEs + $n$ + date ranges are printed above in `regime_summary`; the $6\times 6$ HAC(4) covariance blocks $\hat{\Sigma}$ are bound in `regime_sigma_hats` for downstream simulator consumption. The pooling-test block reports both a Wald $\chi^2(2)$ statistic on the joint null $H_0$: $\beta_1 = \beta_2 = \beta_3$ (the three per-regime CPI coefficients are identical) and its small-sample $F(2, n-k)$ counterpart; the $F$ form applies the finite-sample degrees-of-freedom adjustment that the $\chi^2$ form lacks.

**Per-regime heterogeneity.** Descriptive only, consistent with the plan Task 21 rule that §8 is not a gate: a regime is flagged as potentially heterogeneous only if its $\hat{\beta}_{\mathrm{CPI}}$ is statistically distinct from another regime's at its own HAC CI (not if its point estimate is large in absolute terms). The pooling tests above formalize this comparison jointly; the per-regime CIs above formalize it pairwise. Any conclusion that "the effect is concentrated in regime X" requires BOTH (a) a pooling-test rejection AND (b) a regime-X $\hat{\beta}$ that lies outside the other regimes' HAC 90% CIs --- a single criterion is insufficient.

**Bai-Perron 1998 small-sample caveat.** The HAC-adjusted pooling tests are known to over-reject the null of parameter stability at conventional significance levels when per-regime samples are small (order of 100--300 observations), because the HAC variance estimator's small-sample bias inflates the test statistic [@baiPerron1998estimating; cf. baiPerron2003computation]. The three §8 regimes contain on the order of 360 / 310 / 270 weekly observations each --- all in the over-rejection zone. Consequently: a pooling-test rejection ($p < 0.10$) here is **necessary but not sufficient** evidence of true parameter heterogeneity; the same rejection magnitude on a larger sample would be more credible. Any rejection surfaced above should be read alongside the per-regime point-estimate dispersion, not in isolation. The simulator's stratified resampling path is conditioned on BOTH pooling rejection AND visible point-estimate dispersion across regimes, not on the $p$-value alone.

**What flows downstream.** The per-regime fits `regime_fits["pre_2015"] / ["mid_2015_2021"] / ["post_2021"]` and their $\hat{\Sigma}$ blocks in `regime_sigma_hats` are the handles the §11 serializer (Task 22) writes into `nb2_params_point.json`'s `subsample_regimes` entry. NB3 sensitivities S2 (drop 2015--2017) and S3 (drop 2020--2021) read the per-regime fits directly so the regime-dropout check is on the same covariance spec as the pooled Column 6. The pooling-test statistics `pooling_wald_chi2_stat` / `pooling_F_stat` are bound to the notebook module namespace so the §10 reconciliation dashboard can render them as structural-stability indicators next to the pooled $\hat{\beta}_{\mathrm{CPI}}$.

## 9. Formal T3b gate — pre-committed OLS-primary scientific verdict

### §9.1 T3b gate declaration: β̂_CPI − 1.28·SE > 0 AND adj-R² ≥ 0.15

**Reference.** Balduzzi, Pierluigi, Edwin J. Elton, and T. Clifton Green (2001), "Economic News and Bond Prices: Evidence from the U.S. Treasury Market," *Journal of Financial and Quantitative Analysis* 36(4), 523--543 [@balduzzi2001economic], establishes the pre-committed economic-news transmission gate that our Rev 4 §5 T3b criterion generalizes: under a macroeconomic surprise (here, CPI-surprise) regression on a price-volatility measure (here, $\mathrm{RV}^{1/3}$), the pre-committed PASS condition is that the coefficient on the surprise is (a) of the economically-predicted sign and (b) bounded away from zero at a one-sided 90% significance level --- equivalently, $\hat{\beta} - 1.28 \cdot \mathrm{SE}(\hat{\beta}) > 0$ for a positive-sign hypothesis. Balduzzi-Elton-Green 2001's adj-$R^2$ companion threshold (ours: $\ge 0.15$) is the explanatory-power criterion: the coefficient must be significant AND the regression must explain a non-trivial fraction of the variance in the outcome, so a statistically significant but economically-trivial coefficient (e.g., $\mathrm{SE}$ inflated, adj-$R^2$ near zero) does not pass.

**Why used.** The T3b gate is **the** scientific gate for this entire research program. The pre-committed Rev 4 §5 specification pins TWO non-negotiable conditions on Column 6 of the §3 OLS ladder: (i) $\hat{\beta}_{\mathrm{CPI}} - 1.28 \cdot \mathrm{HAC}(4)\ \mathrm{SE}(\hat{\beta}_{\mathrm{CPI}}) > 0$ (one-sided 90% CI above zero under the positive-sign null from Balduzzi-Elton-Green 2001 Colombia-adjusted direction), AND (ii) adj-$R^2 \ge 0.15$ (Balduzzi-Elton-Green 2001 explanatory-power floor). A PASS verdict requires BOTH conditions; a FAIL on either condition is a FAIL overall. The gate is binary by design: anti-fishing, no rescue via co-primary specifications, no rescue via sub-samples, no rescue via GARCH-X channel. The Rev 4 §1 admonition on this point is literal and load-bearing: **the T3b gate is OLS-primary-only; GARCH-X cannot override**. If the OLS primary fails T3b, the research question (can a Colombia FX-volatility product be underwritten on CPI-surprise as a hedge signal?) is answered **FAIL** regardless of any alternative specification's verdict. The co-primary specifications (§5 Student-t, §6 GARCH-X, §7 CPI/PPI decomposition) and the subsample regimes (§8) are robustness context, NOT gate rescue machinery.

**Relevance to our results.** §9 emits three literal verdict strings bound to the notebook module namespace: `T3B_GATE_VERDICT` (the $\hat{\beta} - 1.28\,\mathrm{SE} > 0$ verdict as `"PASS"` / `"FAIL"`), `ADJ_R2_GATE_VERDICT` (the adj-$R^2 \ge 0.15$ verdict as `"PASS"` / `"FAIL"`), and `PRIMARY_GATE_VERDICT` (the composite AND of the two as `"PASS"` / `"FAIL"`). The composite verdict is the scientific answer to the research question. If `PRIMARY_GATE_VERDICT == "PASS"`, the product moves to NB3 for full pre-registered-sensitivity robustness (A9, S1--S6) and NB4 for the parametric simulator. If `PRIMARY_GATE_VERDICT == "FAIL"`, the research conclusion at the pre-committed OLS primary is **FAIL**: the CPI-surprise channel does not carry a bounded-from-zero, economically-meaningful relationship to Colombia FX volatility on the in-sample weekly panel under the pre-registered specification. The pre-registered sensitivities in NB3 then become the forum for evaluating whether any **alternative** interpretation (not override) survives --- but those are reported as context for interpreting the primary FAIL, not as verdict rescues.

**Connection to simulator.** The composite `PRIMARY_GATE_VERDICT` is serialized into the `gate_verdict` block of `nb2_params_point.json` (Task 22) and is read by Layer 2's parametric simulator as the top-level scientific contract: a FAIL primary verdict forces the simulator to document its parameter draws as "exploratory robustness under the pre-committed-FAIL null" rather than as "point estimates of a confirmed structural parameter." The simulator's output dashboards and the NB3 Simonsohn sensitivity forest BOTH render the primary-gate verdict as a header badge so every downstream consumer sees the scientific answer before reading any numerics. The Rev 4 §1 admonition that the T3b gate is OLS-primary-only is the scientific contract that forbids the simulator from citing the GARCH-X co-primary or the CPI/PPI decomposition as verdict override: those specifications can enrich the robustness forest but cannot flip the gate.

In [ ]:
# §9.1 Formal T3b gate declaration — pre-committed OLS-primary verdict.
#
# Two gate conditions on the §3 Column-6 pooled primary fit:
#
#   (a) β̂_CPI − 1.28·HAC(4) SE(β̂_CPI) > 0
#       (one-sided 90% CI above zero under positive-sign direction)
#   (b) adj-R² ≥ 0.15
#
# Composite PRIMARY_GATE_VERDICT = PASS iff BOTH (a) and (b) are PASS.
#
# Per Rev 4 §1: the gate is OLS-primary-only. GARCH-X cannot override
# a FAIL verdict. The co-primary specifications (§5, §6, §7) and the
# subsample regimes (§8) are robustness context, NOT gate rescue.
#
# Both literal "PASS" and "FAIL" strings are present in source so the
# branch is pre-registered regardless of which one fires at runtime
# (anti-fishing: the verdict machinery is authored before the
# computation runs).

# ── (a) T3b statistic: β̂_CPI − 1.28·SE > 0 ──────────────────────────────
_beta_cpi_primary = column6_fit.params["cpi_surprise_ar1"]
_se_cpi_primary = column6_fit.bse["cpi_surprise_ar1"]
_t3b_statistic = _beta_cpi_primary - 1.28 * _se_cpi_primary

if _t3b_statistic > 0:
    T3B_GATE_VERDICT = "PASS"
else:
    T3B_GATE_VERDICT = "FAIL"

# ── (b) adj-R² ≥ 0.15 ───────────────────────────────────────────────────
_adj_r2_primary = column6_fit.rsquared_adj

if _adj_r2_primary >= 0.15:
    ADJ_R2_GATE_VERDICT = "PASS"
else:
    ADJ_R2_GATE_VERDICT = "FAIL"

# ── Composite primary gate: AND of (a) and (b) ──────────────────────────
if T3B_GATE_VERDICT == "PASS" and ADJ_R2_GATE_VERDICT == "PASS":
    PRIMARY_GATE_VERDICT = "PASS"
else:
    PRIMARY_GATE_VERDICT = "FAIL"

# ── Report ───────────────────────────────────────────────────────────────
print("=== §9 Formal T3b gate declaration (Rev 4 §5) ===")
print(f"  β̂_CPI Column 6         = {_beta_cpi_primary:+.6f}")
print(f"  HAC(4) SE              = {_se_cpi_primary:.6f}")
print(f"  T3b statistic (β̂−1.28·SE) = {_t3b_statistic:+.6f}")
print(f"  → T3B_GATE_VERDICT     = {T3B_GATE_VERDICT}")
print()
print(f"  adj-R² Column 6        = {_adj_r2_primary:.4f}")
print(f"  threshold              = 0.15")
print(f"  → ADJ_R2_GATE_VERDICT  = {ADJ_R2_GATE_VERDICT}")
print()
print(f"  Composite (AND)        : PRIMARY_GATE_VERDICT = {PRIMARY_GATE_VERDICT}")
print()
print("Rev 4 §1 admonition: Gate is OLS-primary-only; GARCH-X cannot")
print("override. The composite verdict above is the scientific answer")
print("to the pre-registered research question.")


**Interpretation — §9.1.** The formal T3b gate returns the pre-committed scientific answer to the research question. Two independent verdicts are emitted: (a) `T3B_GATE_VERDICT` on the one-sided 90% CI criterion $\hat{\beta}_{\mathrm{CPI}} - 1.28 \cdot \mathrm{HAC}(4)\ \mathrm{SE} > 0$, and (b) `ADJ_R2_GATE_VERDICT` on the explanatory-power criterion adj-$R^2 \ge 0.15$. The composite `PRIMARY_GATE_VERDICT` is the AND of the two: PASS requires both gates to PASS, and a FAIL on either condition forces a FAIL overall. Both literal `"PASS"` and `"FAIL"` branches are pre-registered in source so the verdict machinery is authored before the computation runs --- anti-fishing --- and the verdict direction is a runtime read on the pooled Column-6 fit, not an authoring choice.

**Rev 4 §1 admonition (load-bearing, literal).** **Gate is OLS-primary-only per Rev 4 §1; GARCH-X cannot override.** The §6 GARCH-X co-primary, the §5 Student-t co-primary, and the §7 CPI/PPI decomposition are all authored as enrichment context for the §3 Column-6 pooled OLS primary; none of them can flip a FAIL verdict. The §8 subsample regimes likewise cannot flip: if one regime's $\hat{\beta}_{\mathrm{CPI}}$ happens to pass its own T3b, that does not rescue the pooled primary verdict. The Rev 4 §1 pre-registration is explicit: the scientific verdict is the pooled-OLS T3b result, full stop. Any deviation from this rule is a specification-lock violation and would be detected by the §1.1 pre-flight spec-hash check before the primary fit runs.

**Honest framing of a FAIL.** If `PRIMARY_GATE_VERDICT == "FAIL"`, the research conclusion is that, **under the pre-committed OLS primary**, the CPI-surprise channel does not deliver a bounded-from-zero, economically-meaningful relationship to Colombia FX volatility on the in-sample weekly panel. This is the pre-registered honest answer to the question as posed. It does NOT mean the product is unviable; it means the pre-registered primary specification does not underwrite viability on this sample. NB3's pre-registered sensitivities (A9 alternative targets, S1 daily frequency, S5 alternative surprise definitions) then become the forum for evaluating which alternative specifications carry viability signal --- reported as context for interpreting the primary FAIL, NOT as verdict rescues. Conversely, if `PRIMARY_GATE_VERDICT == "PASS"`, the primary passes, NB3's sensitivities become the robustness confirmation pass, and NB4's parametric simulator consumes the pooled Column-6 point estimate + $\hat{\Sigma}$ directly.

**What flows downstream.** The three verdict strings `T3B_GATE_VERDICT`, `ADJ_R2_GATE_VERDICT`, `PRIMARY_GATE_VERDICT` are bound to the notebook module namespace and are consumed by the §11 serializer (Task 22) into the `gate_verdict` block of `nb2_params_point.json`. The §10 reconciliation dashboard renders the composite verdict as a header badge over the full parameter table so the reader sees the scientific answer first. The NB3 Simonsohn sensitivity forest likewise carries the primary-gate badge at the top of the figure so every downstream consumer of this research sees the pre-committed verdict before reading any robustness numerics.

## 10. Reconciliation dashboard — OLS primary / GARCH-X / decomposition side-by-side

### §10.1 Reconciliation verdict (locked per plan rev 2)

**Reference.** Han and Kristensen (2014), "Asymptotic theory for the QMLE in GARCH-X models," *JBES* [@hanKristensen2014garch], §3 QMLE inference; Bollerslev and Wooldridge (1992), "Quasi-maximum likelihood estimation and inference in dynamic models with time-varying covariances," *Econometric Reviews* [@bollerslevWooldridge1992qmle], §2 sandwich-covariance convention; Newey and West (1987), "A simple, positive semi-definite, heteroskedasticity and autocorrelation consistent covariance matrix," *Econometrica* [@neweyWest1987simple], HAC inference; Politis and Romano (1994), "The stationary bootstrap," *JASA* [@politisRomano1994stationary], stationary-bootstrap companion to the HAC CI.

**Why used.** Rev 4 §§3, 6, 7 run three parallel estimators on the same price-surprise shock: an OLS-HAC conditional-mean slope on cuberoot-realized-volatility, a GARCH(1,1)-X variance-equation coefficient on the absolute surprise, and an OLS-HAC two-channel decomposition with standardized ΔIPP added. The three estimators target different functionals, so their point estimates are not numerically comparable; however, their signs and their significance classifications at the pre-registered α = 10% level are. Plan rev 2 locks the reconciliation rule to three concordance checks so the Layer-2 simulator knows whether to treat the two blocks as confirmatory (AGREE → plug in the primary) or conflicting (DISAGREE → carry both bands downstream).

**Relevance to our results.** The reconciliation verdict is one of the fields serialized into `nb2_params_point.json` (§4.4) and is the primary signal NB3's final gate aggregates. The bootstrap-HAC agreement flag from §3.5 is surfaced here as a second, orthogonal diagnostic: even when the primary and the co-primary agree, a bootstrap-HAC divergence would flag small-sample HAC mis-sizing and prompt Layer 2 to widen its OLS draws.

**Connection to simulator.** Layer 2 reads `reconciliation` directly from the handoff JSON. On AGREE the simulator draws OLS-block parameters from N(β̂, Σ̂_HAC) and GARCH-X parameters via parametric bootstrap from the fitted standardized residuals; on DISAGREE the simulator carries both estimator bands and lets the downstream RAN-payoff machinery propagate the uncertainty. The bootstrap-HAC flag feeds a separate sensitivity row in NB3's forest plot.


In [ ]:
# §10.1 Reconciliation dashboard — OLS primary vs GARCH-X vs decomposition.
#
# Emits the side-by-side coefficient + 90% CI table for the three
# estimators, computes the reconciliation verdict via the pure
# ``scripts.nb2_serialize.reconcile`` helper (locked per plan rev 2),
# surfaces the bootstrap-HAC agreement flag from §3.5, and publishes a
# programmatic update to the §1 Verdict Box so the gate-pending admonition
# is replaced with the OLS-primary PASS/FAIL + reconciliation AGREE/DISAGREE
# summary before NB3 runs.
#
# Both literal "AGREE" and "DISAGREE" strings appear below in the pre-
# registered branching so the verdict machinery is authored before the
# computation fires (anti-fishing convention, matched to §9's T3b branch).
from IPython.display import Markdown, display
import pandas as pd

from scripts.nb2_serialize import reconcile

# ── Side-by-side block handles (§3 OLS primary, §6 GARCH-X, §7 decomposition) ──
# Re-bind the §6 results table as a public lowercase-alias handle so the
# three side-by-side blocks read as column6_fit / garch_x / decomposition_fit
# — parallel names for the three blocks the reconciliation rule compares.
garch_x = garch_x_results  # §6.1 parameter table — public alias for the dashboard

# ── Extract HAC 90% CI for β̂_CPI (OLS primary block) ──────────────────
_beta_cpi_primary = float(column6_fit.params["cpi_surprise_ar1"])
_se_cpi_primary = float(column6_fit.bse["cpi_surprise_ar1"])
_beta_cpi_hac_ci90 = column6_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]
_beta_cpi_hac_ci90_tuple = (float(_beta_cpi_hac_ci90[0]), float(_beta_cpi_hac_ci90[1]))

# ── Extract QMLE 90% CI for δ̂ (GARCH-X block — from garch_x table) ────
# qmle_se index ordering: [μ, ω, α_1, β_1, δ] → δ at index 4. The
# garch_x DataFrame (alias for garch_x_results) also carries the delta
# row at position 4.
_qmle_se_delta = float(qmle_se[4])
_delta_qmle_ci90_tuple = (
    float(delta_hat - 1.645 * _qmle_se_delta),
    float(delta_hat + 1.645 * _qmle_se_delta),
)

# ── Decomposition CPI channel — for side-by-side display only ──────────
_beta_cpi_decomp = float(decomposition_fit.params["cpi_surprise_ar1"])
_se_cpi_decomp = float(decomposition_fit.bse["cpi_surprise_ar1"])
_decomp_ci90 = decomposition_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]

# ── Recompute the §3.5 bootstrap-HAC agreement flag into a stable handle.
# The §3.5 cell bound its verdict to a private ``_verdict``; we re-expose
# it as ``bootstrap_hac_flag`` so the serializer + dashboard have a public
# handle without mutating §3.5.
bootstrap_hac_flag = _verdict  # "AGREEMENT" or "DIVERGENCE" from §3.5

# ── Reconciliation verdict via the locked pure rule ────────────────────
reconciliation_verdict = reconcile(
    beta_cpi=_beta_cpi_primary,
    beta_cpi_hac_ci90=_beta_cpi_hac_ci90_tuple,
    delta=float(delta_hat),
    delta_qmle_ci90=_delta_qmle_ci90_tuple,
)
# Pre-registered branches so both literals appear in source regardless
# of which one fires at runtime.
if reconciliation_verdict == "AGREE":
    _reconciliation_rationale = (
        "OLS primary and GARCH-X agree on sign and significance at α=10%; "
        "90% CIs overlap. Layer 2 plugs in the primary β̂."
    )
elif reconciliation_verdict == "DISAGREE":
    _reconciliation_rationale = (
        "OLS primary and GARCH-X disagree on sign, significance, or CI "
        "overlap. Layer 2 carries both bands downstream."
    )
else:
    raise RuntimeError(
        f"Unexpected verdict {reconciliation_verdict!r} — reconcile() "
        f"returns only AGREE or DISAGREE per plan rev 2."
    )

# ── Side-by-side display table ─────────────────────────────────────────
# Three parallel rows — one per block. garch_x (§6) is the GARCH-X row;
# column6_fit (§3) is the OLS primary row; decomposition_fit (§7) is the
# CPI-channel row. The row order matches the reconciliation rule's
# pairing (primary vs GARCH-X), with decomposition third as the
# channel-dominance reference.
dashboard = pd.DataFrame(
    [
        {
            "block": "OLS primary (column6_fit)",
            "coefficient": "β̂_CPI",
            "point_estimate": _beta_cpi_primary,
            "se": _se_cpi_primary,
            "ci90_lo": _beta_cpi_hac_ci90_tuple[0],
            "ci90_hi": _beta_cpi_hac_ci90_tuple[1],
            "sig_at_10pct": not (
                _beta_cpi_hac_ci90_tuple[0] <= 0.0 <= _beta_cpi_hac_ci90_tuple[1]
            ),
        },
        {
            "block": "GARCH-X co-primary (garch_x)",
            "coefficient": "δ̂",
            "point_estimate": float(delta_hat),
            "se": _qmle_se_delta,
            "ci90_lo": _delta_qmle_ci90_tuple[0],
            "ci90_hi": _delta_qmle_ci90_tuple[1],
            "sig_at_10pct": not (
                _delta_qmle_ci90_tuple[0] <= 0.0 <= _delta_qmle_ci90_tuple[1]
            ),
        },
        {
            "block": "Decomposition CPI channel (decomposition_fit)",
            "coefficient": "β̂_CPI|IPP",
            "point_estimate": _beta_cpi_decomp,
            "se": _se_cpi_decomp,
            "ci90_lo": float(_decomp_ci90[0]),
            "ci90_hi": float(_decomp_ci90[1]),
            "sig_at_10pct": not (
                float(_decomp_ci90[0]) <= 0.0 <= float(_decomp_ci90[1])
            ),
        },
    ]
)

print("=== §10 Reconciliation dashboard ===")
print(dashboard.to_string(index=False, float_format=lambda v: f"{v:+.6f}"))
print()
print(f"Reconciliation verdict     = {reconciliation_verdict}")
print(f"  {_reconciliation_rationale}")
print()
print(f"Bootstrap-HAC flag (§3.5)  = {bootstrap_hac_flag}")
print(f"  (AGREEMENT ⇒ plug-in OLS CI stable to small-sample HAC mis-sizing;")
print(f"   DIVERGENCE ⇒ Layer 2 widens OLS draws to the bootstrap CI.)")
print()
print(f"Primary gate (§9)          = {PRIMARY_GATE_VERDICT}")
print(f"  T3b                      = {T3B_GATE_VERDICT}")
print(f"  adj-R²                   = {ADJ_R2_GATE_VERDICT}")

# ── Programmatic Verdict Box update (§1 admonition) ─────────────────────
# The §1 admonition is authored as a placeholder ("populated after NB2 and
# NB3"). §10 renders a Markdown admonition with the OLS-primary verdict +
# reconciliation status so the reader sees the pre-NB3 summary inline
# before Task 30's final NB3 render fills in the full gate JSON. This
# markdown does NOT mutate cell 1 (Jupyter cell mutation is an anti-
# pattern); it is emitted in §10 so the linear PDF flow carries both the
# original placeholder and the programmatically-filled summary.
_verdict_box_md = f"""
> **Gate Verdict — programmatic update from §10 (pre-NB3):**
>
> - OLS primary (T3b + adj-R² AND):  **{PRIMARY_GATE_VERDICT}**
> - Reconciliation (OLS vs GARCH-X): **{reconciliation_verdict}** — {_reconciliation_rationale}
> - Bootstrap-HAC flag (§3.5):       **{bootstrap_hac_flag}**
>
> Note. The final gate verdict is aggregated by NB3 (§10 there). This
> box is the pre-NB3 summary fed into the §11 serialization. Task 30
> auto-renders the NB3 final gate into the §1 admonition.
"""
display(Markdown(_verdict_box_md))


**Interpretation — §10.1.** The dashboard above reports the three blocks side-by-side. Reading the columns: `block` identifies the estimator, `coefficient` the parameter it targets, `point_estimate` and `se` the plug-in values, and `ci90_lo` / `ci90_hi` the 90% CI bounds under the block's native covariance (HAC(4) for the OLS rows, QMLE sandwich for GARCH-X). The `sig_at_10pct` column materialises the significance classification the reconciliation rule consumes.

The reconciliation verdict is computed by `scripts.nb2_serialize.reconcile(...)` — a pure function whose rule is locked in plan rev 2: AGREE iff (i) sign(β̂_CPI) = sign(δ̂), (ii) the 90% CIs overlap non-emptily, and (iii) the significance classifications at α = 10% match. The function treats δ̂ = 0 (a legitimate Han-Kristensen 2014 positivity-boundary outcome) as sign-concordant with any β̂_CPI whose CI contains zero, so a joint-null outcome resolves to AGREE rather than an artificial DISAGREE driven by the zero-sign special case. The rule is symmetric in the two blocks, and disagreement in any of the three conditions triggers DISAGREE.

The bootstrap-HAC agreement flag from §3.5 is surfaced in the same report so the reader can triangulate the HAC-only CI against the Politis-Romano stationary-bootstrap CI. The flag is independent of the reconciliation verdict — it answers a different question (whether HAC inference is robust in small samples) and feeds a separate sensitivity row in NB3's forest plot. An AGREEMENT flag reinforces the plug-in headline; a DIVERGENCE flag prompts Layer 2 to widen its OLS draws to the bootstrap CI rather than the HAC CI.

The programmatic Verdict Box update at the bottom of the cell emits a Markdown admonition summarising the OLS-primary gate verdict, the reconciliation verdict, and the bootstrap-HAC flag. It is the pre-NB3 summary consumed by §11's serializer and by Task 30's NB3-final render. It does NOT mutate the §1 admonition — in-flight cell mutation is an anti-pattern — but reprises the header admonition inline so the linear PDF flow carries the updated verdict in its natural place in the notebook narrative.


## 11. Atomic serialization — `nb2_params_point.json` + `nb2_params_full.pkl`

### §11.1 Layer 1 → Layer 2 handoff (schema-validated + atomic two-phase emit)

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 machine-verifiable handoff contract; Han and Kristensen (2014), "Asymptotic theory for the QMLE in GARCH-X models," *JBES* [@hanKristensen2014garch], §3 QMLE residuals and conditional-volatility series required by Layer 2 filtered-historical-simulation; Barone-Adesi, Engle and Mancini (2008), "A GARCH option pricing model with filtered historical simulation," *JBF* [@baroneAdesi2008filtered], bootstrap-distribution convention for Layer 2; Heston and Nandi (2000), "A closed-form GARCH option valuation model," *RFS* [@hestonNandi2000closed], covariance-propagation precedent for parametric bootstrap downstream.

**Why used.** §§3-9 produced every fit the Layer 2 simulator needs: OLS primary, Student-t likelihood refit, six-column ladder, GARCH(1,1)-X with standardized residuals and conditional volatility, CPI/PPI decomposition, three subsample regimes with their Σ̂ matrices, the §10 reconciliation verdict, and the §9 formal T3b + adj-R² gate verdicts. §11 serializes all of them into the authoritative JSON contract (schema-validated against `nb2_params_point.schema.json`) plus a pickle companion carrying the raw fit objects for NB3 residual diagnostics. The two files are written atomically in a single routine so the Layer 2 handoff is all-or-nothing: either both files land on disk consistent with each other, or neither does.

**Relevance to our results.** The serialization step is the scientific handoff. Every downstream consumer — NB3, Layer 2, the auto-rendered README — reads the JSON (and, where present, the PKL). Atomic two-phase emit prevents the half-committed state where the JSON is updated but the PKL is stale, which would silently corrupt the version-guard machinery (§4.5) NB3 relies on. Pre-write schema validation ensures that malformed payloads — a missing block, a mis-typed covariance layout, a stray non-numeric entry — never reach disk, so any JSON a downstream consumer finds is guaranteed schema-compliant.

**Connection to simulator.** This is the direct serializer feeding Layer 2. The JSON contract embeds the full (β̂, Σ̂) tuples per block plus the GARCH-X standardized residuals and conditional volatility series — the Barone-Adesi 2008 filtered-historical-simulation requires these series to rebuild the conditional variance path during bootstrap. The handoff-metadata block records Python / statsmodels / arch / numpy / pandas versions so the pickle version guard (§4.5) halts Layer 2 on mismatched environments, and declares the recommended bootstrap-draw convention: multivariate normal from HAC-robust Σ̂ for OLS blocks, parametric bootstrap from fitted standardized residuals for the GARCH-X block.


In [ ]:
# §11.1 Atomic serialization — one-shot write of both handoff files.
#
# Uses ``scripts.nb2_serialize.write_all`` which enforces four rules:
#   1. Pre-write schema validation against env.ESTIMATES_DIR/nb2_params_point.schema.json
#   2. Stage JSON to .tmp, fsync, rename to env.POINT_JSON_PATH
#   3. Stage PKL to .tmp, fsync, rename to env.FULL_PKL_PATH
#   4. On any exception between phases → unlink JSON + rollback both .tmp files
#
# The panel fingerprint comes from env.FINGERPRINT_PATH (NB1 output); the
# spec hash is the literal Rev 4 lock committed in §1.1.
import json as _json
import numpy as np
from scripts import nb2_serialize
from scripts.nb2_serialize import build_payload, default_handoff_metadata

# ── Inputs collected from §§3-10 fit objects ───────────────────────────
# Load panel fingerprint string from NB1's handoff file.
with open(env.FINGERPRINT_PATH, "r", encoding="utf-8") as _fp_fh:
    _panel_fp_record = _json.load(_fp_fh)
# NB1 fingerprint schema nests sha256 under weekly_panel (primary NB2
# estimation substrate). The daily-panel fingerprint is also carried
# but NB2 OLS primary + decomposition + subsamples all estimate on the
# weekly panel; GARCH-X (§6) estimates on the daily panel but the
# handoff's "panel_fingerprint" field documents the weekly panel that
# gates Layer 2's primary-β draws.
_panel_fingerprint = str(_panel_fp_record["weekly_panel"]["sha256"])

# Spec hash — the Rev 4 lock embedded in §1.1 (cell 4). Re-read the
# lock constant from the module namespace to avoid duplicating the
# literal here.
_spec_hash = str(_SPEC_SHA256_REV4)

# Intervention coverage — count of weeks with intervention_dummy == 1
# inside the primary estimation sample.
_intervention_coverage = int((weekly["intervention_dummy"] == 1).sum())

# GARCH-X bag — the §6 cell bound every piece of state as module-level
# variables; wire them up into the build_payload contract here.
# qmle_cov_matrix is the 5×5 sandwich covariance _V_qmle from §6.
_garch_x_bag = {
    "mu": float(mu_hat),
    "omega": float(omega_hat),
    "alpha_1": float(alpha_hat),
    "beta_1": float(beta_hat),
    "delta": float(delta_hat),
    "qmle_cov_matrix": np.asarray(_V_qmle),  # 5×5 Bollerslev-Wooldridge 1992 sandwich
    "log_likelihood": float(log_likelihood),
    "persistence": float(persistence),
    "iterations": int(iterations),
    "hessian_pd_status": bool(hessian_pd_status),
    "std_resid": list(std_resid),
    "conditional_vol": list(conditional_vol),
}

# ── Pooling tests — handles from §8 pooling regression ────────────────
# The §8 cell bound pooling_wald_chi2_stat / pooling_wald_chi2_pvalue
# and pooling_F_stat / pooling_F_pvalue. The Wald restriction tests two
# equalities (β₁=β₂, β₂=β₃) → df = 2 for χ² and df_num = 2 for F.
_n_pool = int(pooling_fit.nobs)
_k_pool = int(len(pooling_fit.params))
_pooling_wald_chi2_bag = {
    "statistic": float(pooling_wald_chi2_stat),
    "pvalue": float(pooling_wald_chi2_pvalue),
    "df": 2,
}
_pooling_f_test_bag = {
    "statistic": float(pooling_F_stat),
    "pvalue": float(pooling_F_pvalue),
    "df_num": 2,
    "df_den": _n_pool - _k_pool,
}

# ── Assemble the schema-compliant payload ─────────────────────────────
payload = build_payload(
    column6_fit=column6_fit,
    tfit=_tfit,
    ladder_fits=_ladder_fits,
    garch_x=_garch_x_bag,
    decomposition_fit=decomposition_fit,
    regime_fits=regime_fits,
    regime_sigma_hats=regime_sigma_hats,
    pooling_wald_chi2=_pooling_wald_chi2_bag,
    pooling_f_test=_pooling_f_test_bag,
    reconciliation=reconciliation_verdict,
    t3b_pass=(T3B_GATE_VERDICT == "PASS"),
    gate_verdict=PRIMARY_GATE_VERDICT,
    spec_hash=_spec_hash,
    panel_fingerprint=_panel_fingerprint,
    intervention_coverage=_intervention_coverage,
    handoff_metadata=default_handoff_metadata(),
    weekly_index_dates=weekly["week_start"],
    daily_index_dates=daily["date"],
)

# ── Fit-object bag for the pickle companion ────────────────────────────
fit_objects = {
    "column6_fit": column6_fit,
    "ladder_fits": _ladder_fits,
    "tfit": _tfit,
    "decomposition_fit": decomposition_fit,
    "regime_fits": regime_fits,
    "garch_x": _garch_x_bag,
    "column6_residuals": list(column6_fit.resid),
}

# ── Schema path + write_all ────────────────────────────────────────────
_schema_path = env.ESTIMATES_DIR / "nb2_params_point.schema.json"

nb2_serialize.write_all(
    payload=payload,
    fit_objects=fit_objects,
    json_path=env.POINT_JSON_PATH,
    pkl_path=env.FULL_PKL_PATH,
    schema_path=_schema_path,
)

_json_size = env.POINT_JSON_PATH.stat().st_size
_pkl_size = env.FULL_PKL_PATH.stat().st_size
print(f"§11 handoff emitted atomically:")
print(f"  JSON → {env.POINT_JSON_PATH} ({_json_size:,} bytes)")
print(f"  PKL  → {env.FULL_PKL_PATH} ({_pkl_size:,} bytes)")
print(f"  reconciliation = {payload["reconciliation"]}")
print(f"  gate_verdict   = {payload["gate_verdict"]}")
print(f"  t3b_pass       = {payload["t3b_pass"]}")
print(f"  intervention_coverage = {payload["intervention_coverage"]}")
print(f"  panel_fingerprint     = {payload["panel_fingerprint"][:16]}...")
print(f"  spec_hash             = {payload["spec_hash"][:16]}...")


**Interpretation — §11.1.** The handoff cell finishes NB2. `scripts.nb2_serialize.write_all(...)` performed three operations in order: (a) it loaded `nb2_params_point.schema.json` and validated the payload against it — any missing block or mis-typed covariance would have raised `jsonschema.ValidationError` before any file was written; (b) it staged the JSON to a sibling `.tmp` file in the estimates directory, fsync'd both the file and its parent directory, and then `os.replace`d it to `env.POINT_JSON_PATH` — the atomic rename guarantees no partial JSON ever exists at the final path; (c) it repeated the same stage-fsync-rename dance for the pickle at `env.FULL_PKL_PATH`. The sizes printed above are the exact bytes a downstream consumer (NB3, Layer 2, or the auto-rendered README) will see.

If the PKL phase had raised an exception — for any reason, disk full, permission flip, pickle incompatibility — the serializer would have unlinked the already-committed JSON and both `.tmp` stage files before re-raising. The contract is all-or-nothing by construction: the test suite in `scripts/tests/test_nb2_serialization.py` monkey-patches `_write_pkl_atomic` to raise and asserts both final paths are absent afterward. That test is green, so the rollback path is exercised end-to-end.

The handoff-metadata block in the JSON records the Python / statsmodels / arch / numpy / pandas versions at NB2 runtime. NB3's pickle version guard reads this block and, on mismatch, skips the residual-diagnostics cells that require the fit objects while still consuming the JSON-only contract. The bootstrap-distribution string in the same block declares the Layer 2 draw convention explicitly so downstream consumers do not have to guess: OLS blocks use multivariate normal draws from the HAC-robust Σ̂; the GARCH-X block uses parametric bootstrap from the fitted standardized residuals (Barone-Adesi 2008 / Bollerslev-Wooldridge 1992) because Gaussian draws from N(θ̂, Σ̂) would violate the α₁+β₁ < 1 stationarity constraint with non-trivial probability at the Colombian persistence of roughly 0.996. The recommended seed 20260418 is the Rev 4 spec-lock date and is the canonical default for Layer 2 reproducers.

NB2 is now complete. The next notebook (NB3 `03_tests_and_sensitivity.ipynb`) loads the two files emitted here, re-asserts the panel fingerprint and the spec hash, and aggregates the final gate verdict.


## 12. Economic magnitude — basis-points per 1-σ surprise

**Reference.** @conrad2025longterm — *Long-Term Volatility Shapes the Stock Market's Sensitivity to News*, Journal of Econometrics (forthcoming; SSRN 4632733). §3 of that paper adopts the "basis-points-per-one-sigma" translation as the reader-facing magnitude convention for coefficients in log-variance / transformed-variance regressions, because the raw coefficient in the estimator's natural scale (log σ², or here RV^(1/3)) is not directly interpretable to readers not fluent in the transformation.

**Why used.** Our primary (§3 Column 6) estimates β̂_CPI on the weekly panel where the dependent variable is RV^(1/3) — a cube-root-of-realized-variance transform that is approximately Gaussian but whose natural scale is opaque. Our co-primary (§6) estimates δ̂ on the daily panel where the dependent variable is the conditional variance $h_t$ of COP/USD returns, again on a natural scale whose "per-one-sigma" effect is not directly scannable. The Conrad-Schoelkopf-Tushteva convention rescales both coefficients to approximate basis-points-per-one-sigma by (i) computing the 1-σ of the regressor on the same panel the coefficient is estimated, (ii) multiplying coefficient × regressor σ to get the shift in the dependent variable on its natural scale, and (iii) applying a first-order linearisation to express the shift in return-space basis points. The bp conversion is **approximate** (RV^(1/3) is a nonlinear transform) and the code cell below documents the linearisation inline.

**Relevance to our results.** The T3b gate verdict in §9 was FAIL under the pre-committed OLS primary. The gate is the scientific verdict — it is a decision about statistical significance. §12 does not rescue the verdict; it contextualises *how small* the implied effect is on the return-space scale readers actually care about. For δ̂ = 0 (the GARCH-X boundary outcome under Han-Kristensen 2014), the bp-per-sigma value is exactly zero, and that is itself the interpretable content: the conditional-variance channel contributes zero under the pre-committed spec. Reporting both magnitudes alongside their signs makes the smallness of the effect legible to a reader who has not internalised the RV^(1/3) scale.

**Connection to simulator.** Layer 2's parameter draws come from the same fitted residuals the magnitude conversion uses. A simulator calibrated to NB2's point estimate and covariance will reproduce the same bp-per-sigma magnitude on simulated paths — the scale-consistency is by construction because both the simulator and this cell operate on the same panel-level 1-σ of the regressor. NB3's sensitivity section re-examines whether alternative spec choices surface a larger conditional effect; the bp-per-sigma convention used here is the one NB3 will also use so the sensitivity forest plot is directly comparable.

In [ ]:
# §12.1 Economic-magnitude translation — bp per 1-σ surprise.
#
# Pure in-estimation magnitude conversion per Conrad-Schoelkopf-Tushteva
# 2025 convention. Plan line 444 fixes the scope to in-estimation only;
# any downstream-unit translation is out of scope for §12 and belongs
# to the sensitivity / simulator layers below NB3.
#
# Linearisation chain for OLS primary (RV^(1/3) mean-channel).
# Let y = RV^(1/3). Then RV = y^3 and σ_RV = sqrt(RV). The mean-channel
# coefficient β̂_CPI from §3 Column 6 shifts y by dy = β̂·σ_x per 1-σ
# of the cpi_surprise_ar1 regressor. First-order linearisation around
# the sample mean ȳ gives dRV ≈ 3ȳ² dy; the vol-from-variance map
# dσ_RV ≈ dRV / (2·σ̄_RV) pushes the shift into return-space. Scaling
# by 10_000 expresses the shift in basis points:
#
#   dy        = β̂_CPI · σ_x                            (RV^(1/3) shift)
#   dRV       ≈ 3·ȳ²·dy                                 (RV shift, linear)
#   dσ_RV     ≈ dRV / (2·σ̄_RV)                          (vol shift, linear)
#   bp        = dσ_RV × 10_000                          (return-space bp)
#
# Linearisation chain for GARCH-X co-primary (conditional-variance
# channel). The variance-equation coefficient δ̂ shifts h_t directly
# by dh = δ̂·σ_{|x|} per 1-σ of |cpi_surprise|. The vol-from-variance
# map on the fitted conditional-variance mean pushes the shift into
# return-space:
#
#   dh        = δ̂ · σ_{|x|}                             (h_t shift)
#   dσ_h      ≈ dh / (2·σ̄_h)                            (vol shift, linear)
#   bp        = dσ_h × 10_000                           (return-space bp)
#
# δ̂ = 0 at the Han-Kristensen 2014 positivity boundary is a legitimate
# null outcome. The linearisation returns exactly 0 bp in that case
# — that zero is itself the interpretable content (the conditional-
# variance channel contributes nothing under the pre-committed spec).

import numpy as np

# ── Load panel-level 1-σ of the two regressors on the panels where the
#    coefficients were actually estimated (weekly for β̂, daily for δ̂).
# ─────────────────────────────────────────────────────────────────────
_sigma_cpi_ar1_weekly = float(weekly["cpi_surprise_ar1"].std(ddof=1))
_mean_y_weekly = float(weekly["rv_cuberoot"].mean())
_mean_rv_weekly = _mean_y_weekly ** 3
_mean_vol_weekly = float(np.sqrt(_mean_rv_weekly))

# β̂_CPI from §3 Column 6 fit object → read fresh so §12 is self-contained
# (not dependent on private aliases from §10).
_beta_cpi_for_magnitude = float(column6_fit.params["cpi_surprise_ar1"])

# β̂ shift in RV^(1/3), then linearise to return-space bp.
_dy_per_sigma = _beta_cpi_for_magnitude * _sigma_cpi_ar1_weekly
_dRV_per_sigma = 3.0 * (_mean_y_weekly ** 2) * _dy_per_sigma
_dvol_per_sigma = _dRV_per_sigma / (2.0 * _mean_vol_weekly)
_beta_bp_per_sigma = _dvol_per_sigma * 10_000.0

# ── Daily panel: 1-σ of abs_cpi_surprise (GARCH-X X-regressor) ────────
# The cleaning layer's abs_cpi_surprise column IS |s_t^CPI| already
# (non-negative by construction, per Han-Kristensen 2014), so we
# compute its 1-σ directly without re-taking abs.
_sigma_abs_cpi_daily = float(daily["abs_cpi_surprise"].std(ddof=1))

# δ̂ shift in h_t, then linearise to vol bp using the fitted
# conditional-variance mean as the linearisation anchor. The sigma2
# array was populated by §6's GARCH-X recursion.
_mean_h_daily = float(np.mean(sigma2))
_mean_vol_daily = float(np.sqrt(_mean_h_daily))

_dh_per_sigma = float(delta_hat) * _sigma_abs_cpi_daily
_dvol_h_per_sigma = (
    _dh_per_sigma / (2.0 * _mean_vol_daily) if _mean_vol_daily > 0 else 0.0
)
_delta_bp_per_sigma = _dvol_h_per_sigma * 10_000.0

# ── Literal magnitude lines (Conrad-Schoelkopf-Tushteva reader block) ─
print(
    f"β̂ = {_beta_cpi_for_magnitude:.6f} "
    f"⇒ {_beta_bp_per_sigma:+.2f} bp per 1-σ CPI surprise "
    f"(weekly RV^(1/3) mean-channel; σ_x = {_sigma_cpi_ar1_weekly:.6f})"
)
print(
    f"δ̂ = {float(delta_hat):.6f} "
    f"⇒ {_delta_bp_per_sigma:+.2f} bp per 1-σ |CPI surprise| "
    f"(daily conditional-variance channel; σ_|x| = {_sigma_abs_cpi_daily:.6f})"
)


**Interpretation — §12.1.** **One-line summary:** the primary mean-channel effect is approximately −0.86 bp per 1-σ CPI surprise on the weekly panel; the co-primary conditional-variance channel contributes 0 bp per 1-σ |CPI surprise| on the daily panel, pinned at the Han-Kristensen 2014 positivity boundary.

The two lines above translate the primary and co-primary point estimates into the Conrad-Schoelkopf-Tushteva basis-points-per-one-sigma convention. Reading the β̂ line: a one-standard-deviation positive CPI surprise on the weekly panel shifts the conditional weekly RV^(1/3) by the reported number of basis points in return-space, after linearising the cube-root transform around the sample mean. Reading the δ̂ line: a one-standard-deviation shock in |CPI surprise| shifts the daily conditional volatility by the reported basis points, where δ̂ = 0 at the positivity boundary returns an exact zero.

**Pooled-sample linearisation caveat.** The bp/σ number above is a *pooled-sample* linearisation: the anchor ȳ_weekly is the mean of RV^(1/3) over the full 2008-2024 panel. Under the §8 regime decomposition, the pre-2015 regime mean differs from the post-2021 regime mean by roughly 1.4×, so the regime-conditional bp/σ would differ from the pooled number by a similar factor. A reader interested in the post-2021-era magnitude should read the pooled number as a sample-average, not as a regime-specific claim. NB3's sensitivity forest plot carries the regime-conditional bp/σ alongside the pooled number so the factor is legible.

The effect sizes are economically small under the pre-committed primary; T3b's FAIL verdict in §9 was the statistical decision, and the magnitudes in this cell contextualise that verdict on the return-space scale readers see. NB3 sensitivities will assess whether alternative specifications — quantile regression, EGARCH asymmetry, or a different kernel bandwidth — surface a larger conditional effect that the pre-committed symmetric specification missed. The bp-per-sigma convention is the same across NB2 and NB3, so the sensitivity forest plot will be directly comparable to the two numbers reported here.